# Sesión práctica 1 · Búsqueda semántica en un marketplace

## De localizar palabras a comprender una necesidad

Todas las búsquedas comienzan en la misma caja de texto, pero no todas expresan el mismo tipo de intención. Algunas personas llegan al buscador sabiendo exactamente qué producto quieren. Escriben `taladro 24v batería`, `televisión 28 pulgadas` o el modelo concreto de un dispositivo porque conocen el nombre, la medida o la referencia que debería aparecer en el catálogo.

En esas consultas, las palabras son una señal muy precisa. También lo son los números, las unidades, las marcas y los identificadores de modelo. Si el usuario busca una televisión de 28 pulgadas, devolver una de 55 no constituye una aproximación razonable por mucho que ambos productos pertenezcan a la misma categoría. Un sistema léxico bien construido puede explotar esas coincidencias y resolver este tipo de búsquedas con enorme eficacia.

Sin embargo, muchas personas no saben cómo se llama el producto que necesitan. No formulan la consulta como una referencia de catálogo, sino como una situación que quieren resolver, una restricción que deben cumplir o un resultado que esperan conseguir. Escriben `necesito poner baldas sin hacer agujeros`, `quiero un ordenador que se convierta en tableta` o `busco legumbres aptas para una persona celíaca`.

El catálogo puede contener precisamente los productos adecuados, pero describirlos como *estante sin taladro*, *portátil convertible 2 en 1* y *lentejas sin gluten*. La intención de la consulta y el contenido de la ficha apuntan al mismo objeto, aunque apenas compartan vocabulario. Un buscador que dependa únicamente de coincidencias literales puede no relacionar *sin hacer agujeros* con *sin taladro*, ni *que se convierta en tableta* con *convertible 2 en 1*.

A esta distancia entre las palabras utilizadas por el usuario y las elegidas por el catálogo se la conoce como **vocabulary gap**. No aparece porque el texto se haya tokenizado mal ni porque falte una regla sencilla de normalización. Surge de una propiedad más profunda del lenguaje: una misma necesidad puede expresarse mediante sinónimos, perífrasis, consecuencias, equivalencias entre unidades o conocimiento que ninguna de las dos partes hace explícito.

Una persona puede pedir algo *sin hacer agujeros* donde el vendedor ha escrito *sin taladro*. Puede hablar de *setenta centímetros* cuando la ficha utiliza *28 pulgadas*. Puede describir una silla como adecuada para ofrecer *apoyo para la espalda*, mientras que el catálogo resume esa cualidad con la palabra *ergonómica*. En cada caso, la relación existe en el significado, no necesariamente en la superficie del texto.

Esto no convierte a la búsqueda léxica en una solución obsoleta. Precisamente las consultas con marcas, modelos, medidas, códigos o negaciones muestran por qué sigue siendo necesaria. Un embedding puede reconocer que dos productos son conceptualmente parecidos, pero esa similitud no siempre basta para respetar una referencia exacta, una restricción numérica o una palabra que invierte el sentido de la consulta.

El problema de negocio no consiste, por tanto, en sustituir palabras por vectores. Consiste en construir un ranking capaz de utilizar cada señal donde resulta más fiable: conservar la precisión léxica cuando importan nombres, medidas o referencias, y recuperar por significado cuando la consulta y el catálogo expresan la misma necesidad con vocabularios distintos.

A lo largo de esta sesión estudiaremos las capas que hacen posible esa decisión. Comenzaremos por la representación de consultas y productos, observaremos la geometría que crean los embeddings, analizaremos qué significa medir similitud entre dos vectores y terminaremos evaluando si el ranking recupera realmente los productos que debería. El objetivo no será únicamente ejecutar una búsqueda semántica, sino entender qué información aporta, qué errores puede introducir y cómo combinarla con las señales que ya funcionan bien.

## Índice de contenidos

1. [El problema de negocio y los datos](#1-el-problema-de-negocio-y-los-datos)
2. [La geometría que convierte vectores en rankings](#2-la-geometría-que-convierte-vectores-en-rankings)
3. [Representaciones dispersas](#3-representaciones-dispersas)
4. [Primeras representaciones densas](#4-primeras-representaciones-densas)
5. [Representaciones contextuales](#5-representaciones-contextuales)
6. [Modelos modernos de embeddings](#6-modelos-modernos-de-embeddings)
7. [Más allá de un único vector denso](#7-más-allá-de-un-único-vector-denso)
8. [Evaluación del buscador](#8-evaluación-del-buscador)
9. [Selección de la arquitectura](#9-selección-de-la-arquitectura)


## 1. El problema de negocio y los datos

### 1.1. El contrato de relevancia

Antes de comparar representaciones o algoritmos de búsqueda, necesitamos decidir qué significa que un producto sea relevante para una consulta. Sin ese contrato, cualquier ranking puede parecer razonable y las métricas terminan midiendo una idea de calidad distinta a la que realmente importa para el marketplace.

Trabajaremos con una muestra del **Shopping Queries Dataset** publicado por Amazon Science. El subconjunto local contiene 336 productos del locale español y 12 consultas para las que existen juicios de relevancia. Cada pareja formada por una consulta y un producto recibe una etiqueta ESCI que describe el tipo de relación entre ambos.

La etiqueta `E · Exact` indica que el producto satisface de forma precisa la intención expresada. `S · Substitute` identifica resultados que no coinciden exactamente con lo solicitado, pero que podrían sustituirlo de manera razonable. `C · Complement` se reserva para productos relacionados que acompañan a la necesidad principal sin resolverla por sí mismos. Por último, `I · Irrelevant` señala que el producto no responde a la búsqueda.

Esta escala resulta especialmente útil porque no reduce la relevancia a una decisión binaria. Consideremos la consulta `funda iPad Air 4 sin tapa`. Una funda compatible que incluya tapa puede no respetar completamente la petición, pero todavía funcionar como un sustituto imperfecto. Un protector de pantalla guarda relación con el dispositivo y puede comprarse junto a la funda, aunque no la reemplaza; por tanto, sería un complemento. Un cargador para portátil, en cambio, no resuelve ni acompaña de forma suficientemente directa esa necesidad y se consideraría irrelevante.

Si agrupáramos todas esas relaciones bajo una única etiqueta de relevante o no relevante, perderíamos información importante sobre el ranking. No es lo mismo colocar un sustituto razonable en una posición alta que promover un accesorio complementario por encima de un producto exacto. Las etiquetas ESCI permiten distinguir esos errores y construir métricas que reflejen mejor la utilidad de cada resultado.

Además de las consultas originales, incluiremos ocho **paráfrasis de estrés**. Cada una conserva deliberadamente la intención de una consulta existente, pero modifica las palabras utilizadas para expresarla. Como la necesidad no cambia, la paráfrasis reutiliza el mismo conjunto de productos candidatos y los mismos juicios ESCI que la consulta original.

Estas paráfrasis no pretenden reproducir toda la diversidad de las búsquedas reales ni sustituir una evaluación con tráfico de producción. Su objetivo es mucho más concreto: aislar cuánto depende cada representación de las coincidencias literales. Al mantener fija la intención y cambiar únicamente la superficie léxica, podremos observar qué métodos conservan la calidad cuando desaparecen las palabras más evidentes y cuáles sufren una caída pronunciada.

Comenzaremos preparando el entorno de ejecución y cargando la muestra ESCI. Antes de generar vectores, entrenar modelos o calcular rankings, inspeccionaremos las consultas, los productos asociados, la distribución de etiquetas y las paráfrasis disponibles. De este modo sabremos exactamente qué información recibe cada método y podremos interpretar después sus aciertos y errores sobre ejemplos conocidos.

In [1]:
from pathlib import Path
import os
import sys

current_directory = Path.cwd().resolve()
project_root = next(
    candidate
    for candidate in (current_directory, *current_directory.parents)
    if (candidate / "pyproject.toml").exists()
)
sys.path.insert(0, str(project_root / "src"))


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from dotenv import load_dotenv

load_dotenv(project_root / ".env", override=False)
pio.templates.default = "plotly_white"


In [3]:
from vector_search_session.ecommerce import (
    ESCI_GAINS,
    ESCI_LABEL_NAMES,
    load_esci_sample,
)

sample = load_esci_sample()
products = sample.products.copy()
judgments = sample.judgments.copy()
semantic_queries = pd.read_csv(
    project_root / "data/esci/semantic_queries.csv"
)


In [4]:
unique_queries = (
    judgments[["query_id", "query"]]
    .drop_duplicates()
    .sort_values("query_id", ignore_index=True)
)

print(f"Productos: {len(products):,}")
print(f"Consultas originales: {len(unique_queries):,}")
print(f"Paráfrasis de estrés: {len(semantic_queries):,}")
print(f"Juicios consulta-producto: {len(judgments):,}")


Productos: 336
Consultas originales: 12
Paráfrasis de estrés: 8
Juicios consulta-producto: 336


In [5]:
semantic_queries[
    ["original_query", "semantic_query", "lexical_gap"]
].style.set_properties(subset=["semantic_query"], **{"font-weight": "bold"})


,original_query,semantic_query,lexical_gap
0,convertibles 2 en 1 portátil tactil,quiero un ordenador que pueda transformarse en tableta y manejarse con los dedos,"convertible, 2 en 1 y táctil se expresan mediante una descripción funcional"
1,estantes sin taladro habitacion,necesito poner baldas en el dormitorio sin hacer agujeros en la pared,"estantes, habitación y taladro se sustituyen por sinónimos o consecuencias"
2,funda ipad air 4 sin tapa,quiero proteger por detrás mi iPad Air de cuarta generación dejando la pantalla descubierta,sin tapa se expresa como una restricción sobre la pantalla
3,lentejas sin gluten,busco legumbres que pueda comer una persona celíaca,la propiedad sin gluten se expresa mediante el perfil de la persona
4,sillas oficina ergonomicas,necesito un asiento cómodo para trabajar ocho horas con buen apoyo para la espalda,silla de oficina ergonómica se expresa mediante el problema que resuelve
5,soporte aire acondicionado ventana,cómo puedo fijar el aparato de aire en la ventana para que no se caiga,el sustantivo soporte se convierte en la acción de fijar
6,taladro 24v batería,quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe,taladro y batería se expresan por función y restricción
7,television 28 pulgadas,busco un televisor pequeño de unas setenta centímetros para la cocina,la medida cambia de unidad y se añade el contexto de uso


La tabla anterior hace visible el problema. `Sillas oficina ergonómicas` no es, por sí sola, una demostración convincente de búsqueda semántica: comparte los términos principales con muchas fichas relevantes y un baseline léxico debería funcionar bien. Su pareja `necesito un asiento cómodo para trabajar ocho horas con buen apoyo para la espalda` es distinta. Mantiene la necesidad, pero sustituye la categoría por una situación de uso y los atributos por sus consecuencias.

Esta separación evita una conclusión tramposa. Si un embedding mejora la consulta literal, puede deberse a muchas razones. Si mantiene la calidad cuando se introduce una paráfrasis y TF-IDF cae, se ha observado de forma mucho más limpia la capacidad que se quería estudiar. Aun así, el sistema semántico no recibe un cheque en blanco: también debe sobrevivir a números, negaciones, marcas y productos complementarios.

La evaluación final mantendrá ambas vistas. Las consultas originales representan el comportamiento real publicado en ESCI. Las paráfrasis representan un *slice* adversarial construido para medir robustez semántica. Un modelo solo será una opción razonable si se entiende qué gana en una vista y qué pierde en la otra.

Antes de abandonar los datos, resumiremos ahora cuántos juicios hay de cada clase. La distribución resultante nos permitirá comprobar si la evaluación estará dominada por alguna etiqueta y dará contexto a las métricas del último bloque.


In [6]:
label_summary = (
    judgments["esci_label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="pairs")
)
label_summary["meaning"] = label_summary["label"].map(
    ESCI_LABEL_NAMES
)
label_summary


,label,pairs,meaning
0,E,148,Exact
1,S,105,Substitute
2,I,63,Irrelevant
3,C,20,Complement


In [7]:
label_figure = px.bar(
    label_summary,
    x="label",
    y="pairs",
    color="meaning",
    text="pairs",
    color_discrete_sequence=px.colors.qualitative.Safe,
)
label_figure.update_layout(
    title="Distribución de relevancia graduada en la muestra",
    xaxis_title="Etiqueta ESCI",
    yaxis_title="Pares consulta-producto",
)
label_figure.show()


### 1.2. Qué representa exactamente a un producto

Un modelo de embeddings no recibe un objeto `producto`. Recibe un string. La ficha original separa título, marca, color, bullets y descripción; algunas columnas están vacías y otras repiten información. La función `compose_product_text` convierte esos campos en una representación textual reproducible.

Esta operación no es una limpieza administrativa previa al modelado. Es parte del modelo de recuperación. Repetir tres veces la marca aumenta su presencia en una representación léxica. Anteponer `Marca:` o `Color:` aporta estructura que algunos encoders pueden aprovechar. Incluir una descripción larguísima puede diluir el título, superar el contexto máximo o truncar precisamente la información que importaba.

La versión utilizada concatena título, marca, color, bullets y descripción en ese orden. Si cambia el orden, la plantilla o el tratamiento de nulos, cambia el corpus que se embebe. Por eso un índice reproducible necesita guardar junto a los vectores el identificador del modelo, su dimensión, la normalización y la plantilla exacta de entrada.

Vamos a escoger una ficha concreta, mostrar sus campos originales y compararlos con el texto final que recibirá el encoder. El objetivo es hacer visible una transformación que, de otro modo, quedaría escondida dentro de una función auxiliar.


In [8]:
product_example = products.loc[
    products["product_title"].str.contains("ergon", case=False, na=False)
].iloc[0]

product_example[
    [
        "product_title",
        "product_brand",
        "product_color",
        "product_bullet_point",
        "product_description",
    ]
]


product_title           Silla Racing Vettel en Cuero Sintético & Tela ...
product_brand                                                         CLP
product_color                                              Negro/Amarillo
product_bullet_point    COMODIDAD: La silla de oficina Vettel ofrece u...
product_description     Silla Racing de Oficina Vettel<br><br>La silla...
Name: 75, dtype: str

In [9]:
print(product_example["searchable_text"][:1_200])


Silla Racing Vettel en Cuero Sintético & Tela en Red I Silla Gamer Ergonómica & con Respaldo I Silla de Oficina con Ruedas I Color:, Color:Negro/Amarillo. Marca: CLP. Color: Negro/Amarillo. COMODIDAD: La silla de oficina Vettel ofrece una comodidad envidiable, su forma de diseño ofrece una postura saludable a la hora de sentarse incluso despuésde varias horas de uso. CARACTERÍSTICAS: La silla gamer Vettel tiene un diseño deportivo, esta tapizada en cuero sintético y red, es regulable en altura y tiene un mecanismo de balanceo que permite más flexibilidad en los movimientos. DIMENSIONES: La silla racing Vettel tiene las siguientes medidas: Altura total: 108 - 118 cm | Ancho total: 62 cm | Profundidad total: 68 cm | Altura del asiento: 46 - 56 cm | Superficie del asiento (AxP): 51 x 49 cm | Altura del respaldo: 67 cm | Ancho del respaldo: 50 cm | Altura del reposabrazos: 69 - 79 cm | Peso máximo soportado: 135 kg I Peso: 17 kg. BASE: La silla de escritorio cuenta con un soporte hecho de 

# 2. La geometría que convierte vectores en rankings

Un embedding transforma una entrada —una consulta, un título o una descripción de producto— en un vector de números reales:

$$
f(x)=\mathbf{x}\in\mathbb{R}^{d}.
$$

A partir de ese momento, el sistema deja de comparar textos directamente y pasa a trabajar con posiciones dentro de un espacio. La consulta ocupa un punto, cada producto ocupa otro y la recuperación consiste en decidir cuáles se encuentran más próximos o mejor alineados según una determinada regla de similitud.

Las coordenadas de ese vector no suelen tener una interpretación individual sencilla. En un modelo aprendido, rara vez podemos señalar una dimensión concreta y afirmar que representa *ergonomía*, *color rojo* o *compatibilidad con iPad*. La información se distribuye entre muchas coordenadas y adquiere significado a través de la posición relativa de unos vectores respecto a otros.

Por eso el embedding, por sí solo, todavía no produce un ranking. Después de calcular la representación de la consulta $\mathbf{q}$ y la de cada producto $\mathbf{x}$, necesitamos una segunda función,

$$
s(\mathbf{q},\mathbf{x}),
$$

que convierta la relación geométrica entre ambos vectores en una puntuación. Al ordenar los productos por esa puntuación obtenemos finalmente el ranking que verá el usuario.

Conviene separar estas dos decisiones. El encoder determina qué geometría intenta aprender: qué textos deberían quedar próximos, qué relaciones deberían reflejarse en la dirección de los vectores y qué información puede aparecer también en su magnitud. La función de similitud decide después cómo se interpreta esa geometría durante la búsqueda.

Esa separación importa porque una misma colección de embeddings puede producir rankings distintos según la métrica utilizada. Cambiar de similitud coseno a producto escalar no es una modificación puramente técnica: puede alterar qué productos quedan por delante incluso sin cambiar una sola coordenada. Del mismo modo, normalizar los vectores no es una optimización inocente. Al forzar que todos tengan norma uno, estamos decidiendo que la magnitud deje de influir y que solo importe la dirección.

Para observar estas diferencias con claridad, comenzaremos con un espacio bidimensional deliberadamente interpretable. El eje horizontal representará afinidad con el concepto de *asiento* y el vertical afinidad con *trabajo o ergonomía*. Una silla de oficina podría ocupar una posición alta en ambos ejes, mientras que un taburete se relacionaría con asiento, pero mucho menos con ergonomía.

Un modelo real no trabaja con dos ejes nombrados, sino con cientos o miles de dimensiones cuya interpretación permanece distribuida. Sin embargo, las operaciones geométricas son exactamente las mismas. El producto escalar combina coordenadas, la norma mide la longitud del vector y la distancia compara posiciones, tanto si trabajamos en dos dimensiones como si lo hacemos en 384.

Construiremos primero una consulta y varios productos dentro de este espacio reducido. Los representaremos gráficamente para observar sus direcciones, magnitudes y separaciones, y reutilizaremos el mismo ejemplo en las comparaciones posteriores. Así podremos entender por qué dos métricas aplicadas a los mismos vectores no siempre producen el mismo ranking antes de trasladar esa intuición a embeddings de alta dimensión.

In [10]:
vector_names = [
    "silla ergonómica",
    "silla gaming",
    "silla de comedor",
    "cojín lumbar",
    "mesa de oficina",
]
product_vectors = np.array(
    [[0.95, 0.92], [0.90, 0.62], [0.82, 0.15], [0.38, 0.88], [0.12, 0.74]],
    dtype=np.float32,
)
query_vector = np.array([0.92, 0.90], dtype=np.float32)


In [11]:
vector_figure = go.Figure()
for product_name, product_vector in zip(
    vector_names, product_vectors, strict=True
):
    vector_figure.add_trace(
        go.Scatter(
            x=[0, product_vector[0]],
            y=[0, product_vector[1]],
            name=product_name,
        )
    )


In [12]:
vector_figure.add_trace(
    go.Scatter(
        x=[0, query_vector[0]],
        y=[0, query_vector[1]],
        name="consulta",
        line={"width": 6, "color": "#ef4444"},
    )
)
vector_figure.update_layout(
    title="Consulta y candidatos en un espacio de dos dimensiones",
    xaxis_title="Afinidad con 'asiento'",
    yaxis_title="Afinidad con 'trabajo / ergonomía'",
    height=520,
)
vector_figure.show()


## 2.1. Coseno, producto escalar y distancia euclídea

Una vez fijados los embeddings, todavía queda decidir qué significa que un producto se parezca a la consulta. Esa decisión no es única. Los mismos vectores pueden ordenarse mediante producto escalar, similitud coseno o distancia euclídea, y cada métrica presta atención a una parte distinta de la geometría.

El **producto escalar** multiplica las coordenadas correspondientes y suma los resultados:

$$
\mathbf{q}^{\top}\mathbf{x}
=
\sum_{i=1}^{d} q_i x_i.
$$

Su valor crece cuando la consulta y el producto apuntan en direcciones parecidas, pero también cuando sus vectores tienen una norma grande. Esta doble dependencia se observa al escribirlo como

$$
\mathbf{q}^{\top}\mathbf{x}
=
\lVert\mathbf{q}\rVert_2
\lVert\mathbf{x}\rVert_2
\cos(\theta).
$$

La expresión muestra que el score combina dos factores. Por un lado aparece $\cos(\theta)$, que mide el grado de alineación entre ambos vectores. Por otro, intervienen sus longitudes. Dos productos orientados en una dirección similar pueden obtener puntuaciones distintas si uno de ellos tiene una norma mucho mayor.

Esto puede ser deseable cuando el modelo ha aprendido a utilizar la magnitud como parte de la señal. También puede introducir un sesgo si las normas reflejan factores ajenos a la relevancia, como la longitud del texto, la frecuencia de ciertos patrones o propiedades accidentales del encoder. El producto escalar no distingue entre ambas situaciones: simplemente conserva toda la geometría que recibe.

La **similitud coseno** elimina la influencia de la magnitud al dividir el producto escalar por las normas:

$$
\cos(\theta)
=
\frac{\mathbf{q}^{\top}\mathbf{x}}
{\lVert\mathbf{q}\rVert_2\lVert\mathbf{x}\rVert_2}.
$$

El resultado depende únicamente del ángulo. Dos vectores paralelos obtienen similitud uno aunque uno sea cien veces más largo que el otro. En términos intuitivos, el coseno pregunta si ambos apuntan hacia la misma dirección semántica, no cuánto se alejan del origen.

Esta invariancia resulta útil cuando la longitud del vector no contiene información fiable. Sin embargo, normalizar también implica renunciar a cualquier señal que el modelo hubiera codificado en la norma. Por eso utilizar coseno no es simplemente una forma más estable de calcular similitud: es una decisión acerca de qué parte de la representación consideramos relevante.

La **distancia euclídea** adopta otra perspectiva. En lugar de medir alineación, calcula cuánto hay que desplazarse desde la consulta hasta el producto:

$$
d_{L2}(\mathbf{q},\mathbf{x})
=
\sqrt{
\sum_{i=1}^{d}(q_i-x_i)^2
}.
$$

Dos vectores pueden apuntar en una dirección parecida y, aun así, encontrarse lejos si sus magnitudes son muy distintas. Del mismo modo, dos puntos con normas parecidas pueden tener una distancia grande si ocupan direcciones opuestas. L2 interpreta los embeddings como posiciones completas dentro del espacio, no solo como orientaciones.

También cambia el sentido del ranking. En una similitud, un valor mayor indica normalmente una mejor coincidencia. En una distancia ocurre lo contrario: cuanto menor es el valor, más cerca está el candidato. Una implementación puede convertir la distancia en un score negativo,

$$
s(\mathbf{q},\mathbf{x})=-d_{L2}(\mathbf{q},\mathbf{x}),
$$

para mantener una interfaz que siempre ordene de mayor a menor. Ese cambio de signo modifica la convención del score, pero no altera la geometría ni el orden relativo de los candidatos.

Existe además una relación importante cuando todos los vectores están normalizados a norma uno. En ese caso,

$$
\lVert\mathbf{q}-\mathbf{x}\rVert_2^2
=
2-2\mathbf{q}^{\top}\mathbf{x},
$$

y el producto escalar coincide con la similitud coseno. Ordenar por mayor coseno, mayor producto escalar o menor distancia euclídea produce entonces el mismo ranking. Sin normalización, esa equivalencia desaparece y cada métrica puede favorecer candidatos distintos.

Aplicaremos ahora las tres funciones a los mismos productos. Como los vectores permanecerán fijos, cualquier cambio en el orden podrá atribuirse exclusivamente a la forma en que cada métrica interpreta su dirección, su magnitud y su posición en el espacio.

In [13]:
from vector_search_session.distances import compute_vector_scores

metric_rows = []
for metric_name in ["cosine", "dot", "l2"]:
    metric_scores = compute_vector_scores(
        product_vectors, query_vector, metric=metric_name
    )
    for product_name, score in zip(
        vector_names, metric_scores, strict=True
    ):
        metric_rows.append(
            {"metric": metric_name, "product": product_name, "score": score}
        )


In [14]:
metric_frame = pd.DataFrame(metric_rows)
metric_frame.pivot(index="product", columns="metric", values="score")


metric,cosine,dot,l2
product,,,
cojín lumbar,0.925382,1.1416,0.540370
mesa de oficina,0.804701,0.7764,0.815843
silla de comedor,0.828998,0.8894,0.756637
silla ergonómica,0.999987,1.7020,0.036056
silla gaming,0.985385,1.3860,0.280713


## 2.2. La norma puede cambiar por completo el resultado

Para aislar el papel de la magnitud, imaginemos un candidato que apunta exactamente en la misma dirección que la consulta, pero cuyo vector es cinco veces más largo. Desde el punto de vista angular, ambos representan la misma orientación semántica. Sin embargo, las tres métricas anteriores interpretan esa relación de formas muy distintas.

La similitud coseno no percibe ningún cambio. Como divide por las normas, el factor de escala desaparece y ambos vectores siguen obteniendo el valor máximo:

$$
\cos(\mathbf{q},5\mathbf{q})=1.
$$

Para esta métrica, multiplicar un vector por cinco no modifica su significado relativo. La dirección permanece intacta y la magnitud se considera irrelevante.

El producto escalar sí reacciona al escalado. Al multiplicar uno de los vectores por cinco, su puntuación también se multiplica por cinco:

$$
\mathbf{q}^{\top}(5\mathbf{q})
=
5\mathbf{q}^{\top}\mathbf{q}.
$$

El candidato no está mejor alineado que antes, pero recibe un score mayor porque su norma ha crecido. Si varios productos apuntan en direcciones parecidas, el producto escalar puede favorecer al que tenga mayor magnitud incluso cuando su ángulo no sea el más cercano a la consulta.

La distancia euclídea responde de una tercera manera. La consulta y el candidato se encuentran sobre la misma recta y apuntan hacia el mismo lado, pero ya no ocupan posiciones próximas. La separación entre ambos es

$$
\lVert 5\mathbf{q}-\mathbf{q}\rVert_2
=
4\lVert\mathbf{q}\rVert_2.
$$

Desde la perspectiva de L2, compartir dirección no compensa la distancia absoluta que los separa. El candidato queda lejos porque se ha desplazado mucho respecto al punto concreto ocupado por la consulta.

Estas diferencias no representan defectos matemáticos. Cada métrica responde correctamente a una pregunta distinta. El coseno pregunta si dos vectores apuntan en la misma dirección. El producto escalar combina esa alineación con su magnitud. La distancia euclídea pregunta cuánto separa a los dos puntos dentro del espacio.

El problema aparece cuando la métrica elegida no coincide con el contrato bajo el que se entrenó el modelo. Algunos sistemas de recomendación utilizan deliberadamente el producto escalar y permiten que la norma represente propiedades como popularidad, confianza o intensidad de preferencia. En esos casos, eliminar la magnitud mediante normalización destruiría una parte de la señal aprendida.

Muchos encoders de texto, en cambio, se entrenan o evalúan con similitud coseno y, en ocasiones, entregan directamente vectores normalizados. Para esos modelos, utilizar producto escalar sin normalizar puede introducir un comportamiento que el entrenamiento no pretendía. La elección correcta no depende de qué métrica resulte más familiar, sino de cómo fue construido y validado el espacio de representación.

La norma también puede revelar problemas operativos. Si los vectores asociados a descripciones largas tienden a ser sistemáticamente mayores, el producto escalar podría favorecer productos con más texto aunque su contenido esté peor alineado con la consulta. Algo parecido puede ocurrir si ciertas categorías, idiomas o patrones frecuentes generan distribuciones de magnitud distintas.

Por eso inspeccionar las normas no es una curiosidad geométrica. Conviene estudiar su distribución global, compararla entre segmentos y comprobar si se relaciona con variables que no deberían influir en el ranking. Una cola extrema o diferencias persistentes entre categorías pueden señalar problemas de preprocesamiento, deriva o uso incorrecto de la métrica.

Vamos a observar el efecto de forma controlada. Multiplicaremos la consulta por cinco y repetiremos las comparaciones con los mismos candidatos. La dirección no cambiará; solo aumentará la magnitud. De este modo podremos ver qué métricas permanecen invariantes, cuáles amplifican el score y cuáles interpretan el escalado como un desplazamiento dentro del espacio.

In [15]:
scaled_query = query_vector * 5
extended_vectors = np.vstack([product_vectors, scaled_query])
extended_names = [*vector_names, "consulta escalada x5"]


In [16]:
norm_trap_rows = []
for metric_name in ["cosine", "dot", "l2"]:
    metric_scores = compute_vector_scores(
        extended_vectors, query_vector, metric=metric_name
    )
    for product_name, score in zip(
        extended_names, metric_scores, strict=True
    ):
        norm_trap_rows.append(
            {"metric": metric_name, "product": product_name, "score": score}
        )


In [17]:
pd.DataFrame(norm_trap_rows).pivot(
    index="product", columns="metric", values="score"
)


metric,cosine,dot,l2
product,,,
cojín lumbar,0.925382,1.1416,0.540370
consulta escalada x5,1.000000,8.2820,5.148048
mesa de oficina,0.804701,0.7764,0.815843
silla de comedor,0.828998,0.8894,0.756637
silla ergonómica,0.999987,1.7020,0.036056
silla gaming,0.985385,1.3860,0.280713


## 2.3. Normalización L2 y equivalencia de rankings

La normalización L2 transforma cada vector en otro que conserva su dirección, pero cuya longitud pasa a ser exactamente uno. Para hacerlo, se divide el vector original por su norma:

$$
\widehat{\mathbf{x}}
=
\frac{\mathbf{x}}{\lVert\mathbf{x}\rVert_2}.
$$

Después de esta operación, todos los vectores quedan situados sobre la superficie de una hiperesfera de radio uno. En dos dimensiones sería una circunferencia; en tres, una esfera; y en un espacio de cientos de dimensiones, la misma idea se extiende a una hiperesfera que ya no podemos visualizar directamente.

La normalización elimina cualquier diferencia de magnitud. Dos vectores que apuntan en la misma dirección, aunque uno fuera originalmente cien veces más largo, terminan representados por el mismo punto de la hiperesfera. A partir de ese momento, solo importa el ángulo que forman entre sí.

Esta propiedad hace que la similitud coseno y el producto escalar coincidan exactamente. Si tanto la consulta como el producto tienen norma uno, el denominador del coseno desaparece:

$$
\cos(\widehat{\mathbf{q}},\widehat{\mathbf{x}})
=
\frac{
\widehat{\mathbf{q}}^\top\widehat{\mathbf{x}}
}{
\lVert\widehat{\mathbf{q}}\rVert_2
\lVert\widehat{\mathbf{x}}\rVert_2
}
=
\widehat{\mathbf{q}}^\top\widehat{\mathbf{x}}.
$$

Por tanto, sobre vectores normalizados, ordenar productos por mayor similitud coseno es exactamente lo mismo que ordenarlos por mayor producto escalar. No se trata de una aproximación ni de una coincidencia habitual: ambas puntuaciones toman el mismo valor.

La distancia euclídea también queda ligada a esa misma geometría. Para dos vectores unitarios se cumple

$$
\lVert
\widehat{\mathbf{q}}
-
\widehat{\mathbf{x}}
\rVert_2^2
=
2
-
2\widehat{\mathbf{q}}^\top
\widehat{\mathbf{x}}.
$$

La expresión muestra una relación monótona. Cuando aumenta el producto escalar, disminuye la distancia euclídea al cuadrado. Por eso el candidato con mayor coseno también será el que tenga mayor producto escalar y menor distancia L2.

Las tres métricas producirán así el mismo ranking, aunque sus scores no tengan el mismo valor ni se ordenen en el mismo sentido. Coseno y producto escalar consideran mejor una puntuación alta; L2 considera mejor una distancia baja. La equivalencia afecta al orden de los candidatos, no a la escala utilizada para describirlos.

Esta relación tiene una consecuencia práctica importante. Si los vectores se normalizan una sola vez al generarlos, el coseno puede calcularse después mediante una multiplicación matricial ordinaria. Ya no es necesario volver a calcular la norma de cada candidato para cada consulta. En una colección completa, la matriz de scores puede obtenerse como

$$
S
=
\widehat{Q}
\widehat{X}^{\top},
$$

donde cada fila de $\widehat{Q}$ representa una consulta normalizada y cada fila de $\widehat{X}$ un producto normalizado. Esta operación puede aprovechar implementaciones altamente optimizadas de álgebra lineal.

La normalización debe aplicarse de forma coherente en ambos lados de la búsqueda. Los productos deben normalizarse antes de construir el índice y cada nueva consulta debe normalizarse antes de buscar. Normalizar únicamente una de las dos partes no reproduce el coseno. Aunque una sola norma desaparezca como factor constante para una consulta concreta, la otra continúa influyendo en los scores y puede alterar el ranking.

También hay que considerar el caso de los vectores nulos. Si un encoder devuelve un vector formado únicamente por ceros, su norma es cero y la división no está definida. Este resultado puede aparecer por una entrada vacía, un error de preprocesamiento o una respuesta inválida del modelo. Una implementación robusta debe detectarlo antes de normalizar y evitar que esos valores lleguen al índice como `NaN` o infinitos.

Normalizaremos ahora la consulta y todos los candidatos y volveremos a calcular las tres métricas. Comprobaremos que coseno y producto escalar coinciden numéricamente y que la distancia euclídea induce exactamente el mismo orden cuando se interpreta de menor a mayor. La aserción final convertirá esta propiedad geométrica en una prueba ejecutable: si algún cambio posterior rompe la normalización o el sentido de ordenación, el notebook fallará en lugar de ocultar silenciosamente la inconsistencia.

In [18]:
from vector_search_session.distances import safe_l2_normalize

normalized_products = safe_l2_normalize(extended_vectors, axis=1)
normalized_query = safe_l2_normalize(query_vector)
normalized_cosine = compute_vector_scores(
    normalized_products, normalized_query, metric="cosine"
)
normalized_dot = compute_vector_scores(
    normalized_products, normalized_query, metric="dot"
)


In [19]:
np.testing.assert_allclose(normalized_cosine, normalized_dot, atol=1e-6)
print("Coseno y producto escalar coinciden sobre vectores unitarios.")


Coseno y producto escalar coinciden sobre vectores unitarios.


## 2.4. Dimensionalidad: capacidad, coste y concentración

La dimensión $d$ indica cuántos valores contiene cada embedding. Aumentarla ofrece al modelo más espacio para distribuir información, pero no garantiza que esa capacidad adicional se aproveche de forma útil. También tiene un coste directo: un millón de vectores `float32` de 384 dimensiones ocupan aproximadamente 1,43 GiB antes de añadir cualquier estructura de índice; con 3.072 dimensiones, la cifra asciende a unos 11,44 GiB. El tiempo necesario para calcular productos escalares crece además de forma lineal con $d$.

En espacios aleatorios de alta dimensión aparece un fenómeno conocido como **concentración de distancias**. A medida que aumenta la dimensión, la distancia al vecino más próximo y la distancia al más lejano tienden a parecerse cada vez más en términos relativos. Una forma sencilla de observarlo es calcular la razón entre ambas: cuando la distancia mínima dividida por la máxima se acerca a uno, las distancias separan peor unos puntos de otros.

Esto no significa que un embedding aprendido de alta dimensión vaya a comportarse necesariamente peor. Los vectores producidos por un encoder real no siguen la misma distribución que una nube gaussiana aleatoria. El experimento solo muestra por qué no conviene identificar automáticamente *más dimensiones* con *mejor representación*.

La dimensión adecuada debe elegirse midiendo conjuntamente calidad, memoria y latencia sobre el problema real. Para aislar la intuición geométrica, construiremos ahora nubes aleatorias de dimensión creciente y observaremos cómo evoluciona la relación entre la distancia mínima y la máxima, sin confundir ese resultado con la calidad de un modelo de embeddings.

In [20]:
random_generator = np.random.default_rng(42)
dimension_rows = []
for dimension in [2, 8, 32, 128, 512, 2_048]:
    random_products = random_generator.normal(size=(2_000, dimension))
    random_query = random_generator.normal(size=dimension)
    distances = np.linalg.norm(random_products - random_query, axis=1)
    dimension_rows.append(
        {
            "dimension": dimension,
            "nearest_over_farthest": distances.min() / distances.max(),
        }
    )


In [21]:
dimension_frame = pd.DataFrame(dimension_rows)
dimension_figure = px.line(
    dimension_frame,
    x="dimension",
    y="nearest_over_farthest",
    markers=True,
)
dimension_figure.update_layout(
    title="Concentración de distancias en puntos aleatorios",
    yaxis_title="Distancia mínima / distancia máxima",
)
dimension_figure.show()


# 3. Representaciones dispersas

## 3.1. Bag-of-Words: una coordenada por término

**Bag-of-Words** representa un texto a partir de las palabras que contiene. El primer paso consiste en construir un vocabulario común para todo el corpus. Después se asigna una dimensión distinta a cada término y cada documento se convierte en un vector cuyos valores indican cuántas veces aparece cada palabra.

Si el vocabulario contiene, en este orden, `silla`, `ergonómica`, `mesa` y `roja`, el texto `silla roja roja` se representa como

$$
[1,0,0,2].
$$

La primera coordenada vale uno porque `silla` aparece una vez; la última vale dos porque `roja` aparece dos veces. Las demás permanecen en cero porque esos términos no están presentes en el texto.

Esta representación conserva con bastante precisión el vocabulario utilizado, pero descarta la posición de las palabras. El texto se trata como una bolsa de términos, de ahí su nombre. Por ejemplo, `silla no ergonómica` y `no silla ergonómica` contienen exactamente los mismos unigramas y, por tanto, producen el mismo vector, aunque el orden pueda modificar la interpretación de la frase.

La pérdida de orden no impide que Bag-of-Words resulte útil. En recuperación de información, la presencia de términos concretos suele ser una señal muy valiosa. Marcas, modelos, materiales, medidas o referencias técnicas pueden identificarse directamente mediante las dimensiones que les corresponden. El problema aparece cuando la relevancia depende de relaciones entre palabras o cuando la consulta y el producto expresan la misma idea mediante vocabularios distintos.

El tamaño del vector viene determinado por el vocabulario, no por la longitud del documento. Si el corpus contiene 50.000 términos distintos, cada producto queda conceptualmente representado por un vector de 50.000 dimensiones. Sin embargo, un producto individual solo utiliza una fracción diminuta de ellas. Un texto con 40 términos distintos activaría, como máximo, 40 coordenadas y dejaría aproximadamente un 99,92 % del vector en cero.

Guardar esa matriz como si todos sus valores fueran necesarios supondría desperdiciar una enorme cantidad de memoria. Por eso las representaciones Bag-of-Words no suelen almacenarse como matrices densas. Se utilizan formatos dispersos como **CSR** (*Compressed Sparse Row*), que conservan únicamente la información necesaria para reconstruir los valores no nulos.

En lugar de almacenar miles de ceros por documento, CSR mantiene tres estructuras: los valores distintos de cero, las columnas en las que aparecen y unos punteros que indican dónde comienza y termina cada fila. Para un producto que solo contiene unos pocos términos, el almacenamiento queda así ligado al número de palabras activas y no al tamaño completo del vocabulario.

Esto aclara una distinción importante. Una representación es **dispersa** por la proporción de coordenadas iguales a cero, no porque tenga pocas dimensiones. De hecho, los vectores Bag-of-Words suelen tener una dimensionalidad muy alta. Un embedding de 384 dimensiones, en cambio, puede ser mucho más pequeño pero considerarse denso porque prácticamente todas sus coordenadas contienen algún valor.

La diferencia afecta a mucho más que al formato de almacenamiento. También determina qué operaciones resultan eficientes y qué estructuras de recuperación pueden utilizarse. Las representaciones dispersas encajan de forma natural con índices invertidos y operaciones que recorren únicamente los términos presentes. Los embeddings densos requieren normalmente productos escalares, distancias vectoriales e índices diseñados para espacios continuos.

Construiremos ahora la matriz Bag-of-Words del catálogo y examinaremos su forma, el tamaño del vocabulario y el porcentaje real de ceros. También observaremos cuánto espacio ocupa su representación dispersa frente a lo que requeriría una matriz densa equivalente. De esta forma, `sparse` dejará de ser una etiqueta abstracta y se convertirá en una propiedad medible de nuestros datos.

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 1),
    min_df=2,
)
count_matrix = count_vectorizer.fit_transform(
    products["searchable_text"]
)


In [23]:
total_positions = count_matrix.shape[0] * count_matrix.shape[1]
density = count_matrix.nnz / total_positions
csr_bytes = (
    count_matrix.data.nbytes
    + count_matrix.indices.nbytes
    + count_matrix.indptr.nbytes
)
dense_bytes = total_positions * np.dtype(np.int64).itemsize

print(f"Matriz: {count_matrix.shape}")
print(f"Densidad: {density:.2%}")
print(f"CSR: {csr_bytes / 2**20:.2f} MiB")
print(f"Densa equivalente: {dense_bytes / 2**20:.2f} MiB")


Matriz: (336, 3642)
Densidad: 2.79%
CSR: 0.39 MiB
Densa equivalente: 9.34 MiB


## 3.2. TF-IDF: distinguir una coincidencia informativa de una trivial

Bag-of-Words permite saber qué términos aparecen en cada documento y con qué frecuencia, pero trata todas las palabras como si aportaran la misma información. Esto puede producir rankings poco útiles: un término repetido muchas veces recibe un peso alto aunque aparezca en casi todo el catálogo, mientras que una referencia muy específica puede quedar infravalorada pese a ser mucho más discriminativa.

**TF-IDF** corrige ese problema combinando dos señales. La frecuencia de término, $tf(t,d)$, mide cuánto aparece el término $t$ dentro del documento $d$. La frecuencia inversa de documento, $idf(t)$, reduce el peso de las palabras presentes en gran parte de la colección:

$$
idf(t)
=
\log\left(
\frac{N+1}{df(t)+1}
\right)+1.
$$

Aquí, $N$ representa el número total de documentos y $df(t)$ cuántos de ellos contienen el término. Cuanto más extendida está una palabra por el catálogo, menor es su capacidad para distinguir unos productos de otros y, por tanto, menor será su IDF.

Por ejemplo, `iPad Air 4` puede ser una señal muy fuerte si solo aparece en unas pocas fichas. En cambio, términos como `producto`, `calidad` o `para` pueden estar presentes en miles de descripciones y apenas ayudan a decidir qué resultado responde mejor a una consulta concreta. TF-IDF no elimina necesariamente esas palabras, pero limita su influencia frente a términos más específicos.

También conviene controlar el efecto de la repetición. Sin ninguna transformación, escribir una palabra veinte veces multiplicaría por veinte su frecuencia y podría dominar el vector. Con `sublinear_tf=True`, la frecuencia se comprime aproximadamente como

$$
1+\log(tf),
$$

de modo que repetir un término sigue aumentando su peso, pero cada repetición adicional aporta menos que la anterior. La primera aparición importa mucho más que pasar de diecinueve a veinte.

`TfidfVectorizer` normaliza además cada fila con norma L2. Esto tiene una consecuencia práctica importante: una vez normalizados los documentos y la consulta, el producto escalar entre ambos coincide con la similitud coseno. Por tanto, podemos calcular todos los scores mediante una multiplicación entre la matriz dispersa de documentos y el vector columna de la consulta, sin convertir la representación a formato denso.

Sustituiremos ahora los conteos de Bag-of-Words por pesos TF-IDF aprendidos exclusivamente sobre el catálogo. Después inspeccionaremos los términos con mayor IDF para comprobar qué palabras considera especialmente discriminativas esta colección concreta. Esa observación también nos permitirá detectar si el vocabulario contiene referencias útiles, términos excesivamente raros o posibles artefactos de preprocesamiento.

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 1),
    sublinear_tf=True,
    norm="l2",
)
tfidf_matrix = tfidf_vectorizer.fit_transform(
    products["searchable_text"]
)
print(f"Shape TF-IDF: {tfidf_matrix.shape}")


Shape TF-IDF: (336, 7136)


## 3.3. La consulta debe atravesar exactamente la misma transformación

Cuando construimos la representación TF-IDF del catálogo, `fit_transform` realiza dos tareas al mismo tiempo: aprende qué términos forman parte del vocabulario y calcula el valor IDF de cada uno a partir de su presencia en los documentos. Ese espacio queda fijado en ese momento.

Las consultas nuevas no deben volver a aprenderlo. En su lugar, atraviesan `transform`, que utiliza exactamente el mismo vocabulario, los mismos pesos IDF y las mismas reglas de normalización. Si una consulta contiene una palabra que nunca apareció en el catálogo durante el ajuste, esa palabra no tiene una coordenada asociada y, por tanto, no contribuye al vector.

Esta asimetría es necesaria. Los documentos ya están representados dentro de un espacio concreto; ampliar el vocabulario o recalcular los pesos con cada consulta cambiaría ese espacio y haría que las representaciones almacenadas dejaran de ser comparables. En producción, el vectorizador forma parte del artefacto del índice y debe versionarse junto a él.

La diferencia entre una consulta literal y una paráfrasis permite observar claramente esta limitación. `television 28 pulgadas` comparte con el catálogo tanto la categoría como una medida exacta. Si esos términos aparecen en las fichas adecuadas y tienen un peso suficientemente discriminativo, TF-IDF debería recuperar buenos resultados.

La consulta `busco un televisor pequeño de unos setenta centímetros para la cocina` expresa prácticamente la misma necesidad, pero cambia varias señales a la vez. Utiliza `televisor` en lugar de `television`, convierte la medida a centímetros y sustituye parte de la especificación por una cualidad más vaga como `pequeño`. TF-IDF puede aprovechar cualquier término compartido, pero no sabe por sí mismo que 28 pulgadas equivalen aproximadamente a 71 centímetros ni que esa medida suele describir un televisor de tamaño reducido.

Esa incapacidad semántica no convierte a TF-IDF en un baseline débil. Su literalidad también protege información importante. El sistema no presupone que `24V` y `18V` sean intercambiables, ni que una funda `con tapa` satisfaga una consulta que pide explícitamente `sin tapa`. Allí donde una coincidencia exacta importa, esa falta de imaginación puede convertirse en una ventaja.

El objetivo de las representaciones posteriores no será borrar la señal léxica, sino complementarla. La búsqueda semántica puede ayudar cuando cambia el vocabulario, mientras que TF-IDF conserva una precisión especialmente valiosa para modelos, medidas, marcas, referencias y negaciones.

Compararemos ahora una consulta literal con su paráfrasis semántica. Ambas pasarán por el mismo `transform` y se evaluarán sobre la misma matriz TF-IDF del catálogo. De este modo, cualquier diferencia en el ranking procederá únicamente de cómo se ha expresado la intención, no de cambios en el vocabulario, los pesos o el índice.

In [25]:
demonstration_query_id = 101_352
original_text = semantic_queries.loc[
    semantic_queries["query_id"] == demonstration_query_id,
    "original_query",
].iloc[0]
semantic_text = semantic_queries.loc[
    semantic_queries["query_id"] == demonstration_query_id,
    "semantic_query",
].iloc[0]


In [26]:
original_tfidf = tfidf_vectorizer.transform([original_text])
semantic_tfidf = tfidf_vectorizer.transform([semantic_text])
original_scores = (tfidf_matrix @ original_tfidf.T).toarray().ravel()
semantic_scores = (tfidf_matrix @ semantic_tfidf.T).toarray().ravel()


In [27]:
def top_products(
    scores: np.ndarray,
    limit: int = 6,
    query_id: int | None = None,
) -> pd.DataFrame:
    top_indices = np.argsort(-scores, kind="stable")[:limit]
    result = products.iloc[top_indices][
        ["product_id", "product_title", "product_brand"]
    ].copy()
    result["score"] = scores[top_indices]
    if query_id is not None:
        query_labels = judgments.loc[
            judgments["query_id"] == query_id,
            ["product_id", "esci_label"],
        ]
        result = result.merge(query_labels, on="product_id", how="left")
    return result


In [28]:
pd.concat(
    {
        "consulta literal": top_products(
            original_scores, query_id=demonstration_query_id
        ),
        "paráfrasis": top_products(
            semantic_scores, query_id=demonstration_query_id
        ),
    },
    names=["entrada", "fila"],
)


product_id  \
entrada          fila               
consulta literal 0     B07QVHD5BS   
                 1     B07CQ84VKG   
                 2     B07CRRY36Y   
                 3     B07WV3V2GV   
                 4     B000IHYS2Q   
                 5     B07SC6Z95F   
paráfrasis       0     B087QYF1WC   
                 1     B00KKFCLF2   
                 2     B07SWYRYB4   
                 3     B071DXH7YQ   
                 4     B087Q5SLQT   
                 5     B091T4NPSM   

                                                           product_title  \
entrada          fila                                                      
consulta literal 0     TV Televisión Televisor Engel Ever-LED LE2050 ...   
                 1     LG Electronics 28TK410V-PZ - Monitor/TV de 28"...   
                 2     LG Consumer Electronics 28TK410V-WZ - Monitor ...   
                 3     TV LED 28" INFINITON HD Ready - HDMI, 500Hz, M...   
                 4     Fujifilm FinePix S6500 - Cámara Digital Compac...   
                 5               Nevir NVR-7702-28RD2-N - TV, Multicolor   
paráfrasis       0     Pipishell Baldas Pared, Estanteria de Pared de...   
                 1     SONGMICS Taburete de Bar Juego de 2, Sillas de...   
                 2     Vencipo Estantería Blanca para Fotos de Colgan...   
                 3     Zollner Taburete plegable multiusos, 30 cm altura   
                 4     Piorlado Baldas Pared Negro de 3 Niveles, Esta...   
                 5     DUFU Estanteria Ducha de Esquina para Baño Sin...   

                              product_brand     score esci_label  
entrada          fila                                             
consulta literal 0               Engel Axil  0.253080          S  
                 1                       LG  0.124567          E  
                 2                       LG  0.122187          E  
                 3     INFINITON ELECTRONIC  0.104525          E  
                 4                 Fujifilm  0.091560        NaN  
                 5                    Nevir  0.083894          S  
paráfrasis       0                Pipishell  0.123402        NaN  
                 1                 SONGMICS  0.105485        NaN  
                 2                  Vencipo  0.103417        NaN  
                 3                  ZOLLNER  0.102513        NaN  
                 4                 piorlado  0.101748        NaN  
                 5                     DUFU  0.098948        NaN

## 3.4. BM25: un ranking léxico pensado para recuperar

TF-IDF nos ha permitido construir un baseline léxico muy útil: cada producto y cada consulta ocupan el mismo espacio disperso, y el producto escalar entre vectores normalizados ordena las coincidencias. Sin embargo, los buscadores léxicos que se utilizan habitualmente en producción suelen puntuar esas coincidencias con **BM25**. No porque TF-IDF haya dejado de ser importante, sino porque BM25 modela de forma más realista dos fenómenos que aparecen continuamente en un catálogo: repetir una palabra no debe aumentar su relevancia sin límite y un documento largo no debería recibir ventaja únicamente por contener más términos.

BM25 no es un embedding alternativo ni produce un vector denso que después comparemos mediante coseno. Es una función de ranking que se aplica a una pareja formada por consulta y documento. Para cada término compartido, combina su rareza en la colección, su frecuencia dentro de la ficha y la longitud total de esa ficha:

$$
\operatorname{BM25}(q,d)
=
\sum_{t\in q}
idf(t)
\frac{tf(t,d)(k_1+1)}{tf(t,d)+k_1\left(1-b+b\frac{|d|}{\overline{|d|}}\right)}.
$$

El parámetro $k_1$ controla cuánto tarda en saturarse la frecuencia de término. Una segunda aparición de `television` puede aportar señal; la vigésima difícilmente debería multiplicar por veinte la relevancia. El parámetro $b$ controla la normalización por longitud. Una descripción extensa tiene más oportunidades de coincidir con una query por puro volumen de texto, y BM25 compensa parte de esa ventaja comparándola con la longitud media del catálogo.

En un motor de búsqueda, esta función se ejecuta sobre un **índice invertido**: para cada término se conserva la lista de documentos que lo contienen y la frecuencia con la que aparece en ellos. Así, una consulta no recorre todas las fichas del catálogo; visita las listas de postings de sus términos activos. En este notebook implementaremos el cálculo sobre una matriz dispersa porque deja visible cada componente de la fórmula. El objetivo no es construir otro motor, sino entender por qué BM25 es una referencia léxica más realista que un coseno TF-IDF cuando se habla de recuperación en producción.

Para que la comparación sea limpia, BM25 utilizará el mismo texto de búsqueda de cada producto y las mismas reglas básicas de tokenización que ya empleamos en Bag-of-Words. A partir de los conteos obtendremos la longitud de cada ficha, la longitud media del catálogo, la frecuencia de documento de cada término y un IDF suavizado. En una implementación productiva, esas estadísticas y los postings formarían parte del índice léxico persistido.

In [29]:
bm25_vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 1),
)
bm25_count_matrix = bm25_vectorizer.fit_transform(
    products["searchable_text"]
).tocsr()

bm25_document_lengths = np.asarray(
    bm25_count_matrix.sum(axis=1)
).ravel()
bm25_average_document_length = bm25_document_lengths.mean()
bm25_document_frequency = np.asarray(
    (bm25_count_matrix > 0).sum(axis=0)
).ravel()

bm25_idf = np.log1p(
    (len(products) - bm25_document_frequency + 0.5)
    / (bm25_document_frequency + 0.5)
)

print(f"Vocabulario BM25: {bm25_count_matrix.shape[1]:,} términos")
print(f"Longitud media: {bm25_average_document_length:.1f} términos")


Vocabulario BM25: 7,136 términos
Longitud media: 213.1 términos


Con esas estadísticas ya podemos puntuar una consulta. La función siguiente recorre únicamente los términos que sobreviven a la tokenización de la query. Para cada uno, obtiene su frecuencia en las fichas del catálogo y aplica la saturación y la corrección por longitud de BM25. Esta forma matricial es deliberadamente transparente; un índice invertido real evitaría materializar una columna completa para cada término.

In [30]:
def bm25_scores(
    query_text: str,
    *,
    k1: float = 1.5,
    b: float = 0.75,
) -> np.ndarray:
    query_counts = bm25_vectorizer.transform([query_text])
    length_normalizer = 1 - b + b * (
        bm25_document_lengths / bm25_average_document_length
    )
    scores = np.zeros(len(products), dtype=np.float64)

    for term_index in query_counts.indices:
        term_frequency = bm25_count_matrix[:, term_index].toarray().ravel()
        numerator = term_frequency * (k1 + 1)
        denominator = term_frequency + k1 * length_normalizer
        scores += bm25_idf[term_index] * numerator / denominator

    return scores


Aplicaremos ahora BM25 a la misma pareja de consultas sobre televisores. La consulta literal contiene varios términos concretos que deberían activar postings muy informativos. La paráfrasis conserva la necesidad, pero cambia la formulación, la unidad y parte del vocabulario. BM25 puede refinar el ranking de coincidencias existentes; no sabe por sí solo que `setenta centímetros` y `28 pulgadas` describen aproximadamente la misma medida.

Los valores BM25 no viven en la misma escala que los scores de TF-IDF ni que los cosenos de los embeddings. La comparación útil no consiste en restarlos, sino en leer el orden de los candidatos y comprobar qué productos llegan al ranking.

In [31]:
bm25_original_scores = bm25_scores(original_text)
bm25_semantic_scores = bm25_scores(semantic_text)

pd.concat(
    {
        "BM25 · consulta literal": top_products(
            bm25_original_scores,
            query_id=demonstration_query_id,
        ),
        "BM25 · paráfrasis": top_products(
            bm25_semantic_scores,
            query_id=demonstration_query_id,
        ),
    },
    names=["entrada", "fila"],
)


product_id  \
entrada                 fila               
BM25 · consulta literal 0     B07QVHD5BS   
                        1     B07WV3V2GV   
                        2     B07CQ84VKG   
                        3     B07CRRY36Y   
                        4     B000IHYS2Q   
                        5     B07XM84G9L   
BM25 · paráfrasis       0     B087QYF1WC   
                        1     B00KKFCLF2   
                        2     B087Q5SLQT   
                        3     B07SWYRYB4   
                        4     B071DXH7YQ   
                        5     B091T4NPSM   

                                                                  product_title  \
entrada                 fila                                                      
BM25 · consulta literal 0     TV Televisión Televisor Engel Ever-LED LE2050 ...   
                        1     TV LED 28" INFINITON HD Ready - HDMI, 500Hz, M...   
                        2     LG Electronics 28TK410V-PZ - Monitor/TV de 28"...   
                        3     LG Consumer Electronics 28TK410V-WZ - Monitor ...   
                        4     Fujifilm FinePix S6500 - Cámara Digital Compac...   
                        5     Nuevo Vestido Medieval para Mujer Adulto gótic...   
BM25 · paráfrasis       0     Pipishell Baldas Pared, Estanteria de Pared de...   
                        1     SONGMICS Taburete de Bar Juego de 2, Sillas de...   
                        2     Piorlado Baldas Pared Negro de 3 Niveles, Esta...   
                        3     Vencipo Estantería Blanca para Fotos de Colgan...   
                        4     Zollner Taburete plegable multiusos, 30 cm altura   
                        5     DUFU Estanteria Ducha de Esquina para Baño Sin...   

                                     product_brand      score esci_label  
entrada                 fila                                              
BM25 · consulta literal 0               Engel Axil  12.574036          S  
                        1     INFINITON ELECTRONIC   7.958418          E  
                        2                       LG   7.723772          E  
                        3                       LG   7.707070          E  
                        4                 Fujifilm   7.054629        NaN  
                        5              MUJER FELIZ   5.779584        NaN  
BM25 · paráfrasis       0                Pipishell   9.082614        NaN  
                        1                 SONGMICS   8.750584        NaN  
                        2                 piorlado   8.697288        NaN  
                        3                  Vencipo   8.374481        NaN  
                        4                  ZOLLNER   8.041279        NaN  
                        5                     DUFU   7.319552        NaN

BM25 no elimina el *vocabulary gap*: sigue dependiendo de que consulta y ficha compartan términos que el analizador pueda reconocer. Su valor está en tratar esa señal compartida con una función de ranking diseñada para la recuperación léxica y ejecutable sobre índices invertidos. Más adelante, cuando combinemos una señal léxica con otra densa, BM25 será una candidata natural para conservar marcas, modelos, códigos, medidas y restricciones exactas.

El cambio de ranking no debe interpretarse únicamente a partir de los scores. Conviene leer también los títulos recuperados y comprobar qué coincidencias han llevado a cada producto hasta esas posiciones. En TF-IDF, una puntuación igual a cero indica que la consulta y el documento no comparten ninguna característica activa; un valor alto señala un solapamiento léxico importante después de ponderar los términos por su capacidad discriminativa.

Ese score no representa una probabilidad de compra ni un porcentaje de relevancia. Tampoco permite afirmar que un producto con 0,8 sea “el doble de relevante” que otro con 0,4. Su función es ordenar candidatos dentro del espacio construido por el vectorizador.

Los modelos densos abordarán el problema desde otra perspectiva. Intentarán situar cerca expresiones como *televisión de 28 pulgadas*, *televisor pequeño* y *pantalla de unos setenta centímetros*, aunque no compartan exactamente las mismas palabras. No realizan una conversión de unidades fiable como lo haría una calculadora; aprenden asociaciones distribucionales a partir de ejemplos y pueden cerrar parte de esa distancia semántica.

Para entender de dónde procede esa capacidad, recorreremos ahora la evolución desde los vectores de palabra estáticos hasta los encoders contrastivos de frases. Ese recorrido mostrará qué información conserva cada representación, qué limitaciones resuelve y qué errores nuevos puede introducir.

# 4. Primeras representaciones densas

## 4.1. Word2Vec: aprender una palabra por los contextos que la rodean

Word2Vec no necesita un diccionario de sinónimos para aprender que dos palabras están relacionadas. Su punto de partida es una idea distribucional: las palabras que aparecen en contextos parecidos suelen desempeñar funciones semánticas parecidas. Para capturar esa regularidad, el modelo recorre el corpus mediante ventanas y construye ejemplos a partir de las palabras que aparecen próximas entre sí.

En la variante **Skip-gram**, la palabra central $w_t$ se utiliza para predecir las palabras de contexto $w_{t+j}$ que la rodean. **CBOW** plantea la tarea inversa: combina las palabras del contexto e intenta predecir cuál ocupaba el centro. Aunque ambas variantes parten de los mismos textos, organizan la señal de entrenamiento de forma distinta.

Calcular una probabilidad sobre todo el vocabulario mediante softmax resultaría costoso en cada ejemplo. **Negative sampling** sustituye esa normalización completa por una tarea binaria más sencilla. Para una pareja observada $(w,c)$, el modelo intenta aumentar

$$
\log \sigma(\mathbf{v}_c^\top \mathbf{v}_w),
$$

de modo que sus vectores produzcan un producto escalar alto. Al mismo tiempo, para varios contextos negativos $n$ que no aparecieron junto a la palabra central, maximiza

$$
\log \sigma(-\mathbf{v}_n^\top \mathbf{v}_w),
$$

empujando esas parejas en la dirección contraria. Como las palabras que comparten contextos reciben actualizaciones semejantes, sus vectores terminan ocupando regiones próximas del espacio.

El resultado final es una tabla con un vector por entrada del vocabulario. Esa representación puede capturar relaciones distribucionales útiles, pero sigue siendo estática: el vector de `banco` será siempre el mismo, tanto si la palabra aparece en una consulta sobre una entidad financiera como si describe un asiento. Word2Vec aprende un significado agregado a partir de todos sus usos, pero no adapta la representación a cada contexto concreto.

Tampoco genera directamente un vector para una consulta o una ficha de producto completa. Una solución habitual consiste en promediar los vectores de sus palabras, aunque se trata de una heurística limitada: pierde el orden, mezcla todos los términos en una única representación y les asigna la misma importancia salvo que se introduzca alguna ponderación adicional.

Entrenaremos un pequeño modelo Skip-gram sobre las fichas del catálogo y consultaremos los vecinos de `silla`. El corpus es deliberadamente reducido y no debería producir un embedding competitivo. El objetivo es observar cómo surge la proximidad entre palabras a partir de sus contextos y reconocer las limitaciones que motivaron las representaciones posteriores.

In [32]:
from gensim.models import FastText, Word2Vec
from gensim.utils import simple_preprocess

tokenized_products = [
    simple_preprocess(product_text, deacc=True)
    for product_text in products["searchable_text"]
]


In [33]:
word2vec_model = Word2Vec(
    sentences=tokenized_products,
    vector_size=64,
    window=5,
    min_count=2,
    sg=1,
    negative=8,
    workers=1,
    seed=42,
    epochs=40,
)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [34]:
word2vec_model.wv.most_similar("silla", topn=8)


[('racing', 0.7517865896224976),
 ('gaming', 0.7327979803085327),
 ('claudia', 0.7194018959999084),
 ('vettel', 0.7064101696014404),
 ('ergonomica', 0.7022814750671387),
 ('gamer', 0.6979247331619263),
 ('brazos', 0.6927688121795654),
 ('sillon', 0.6849839687347412)]

El entrenamiento anterior utiliza únicamente 336 productos, de modo que sus vecinos deben interpretarse con cautela. No representan la calidad que Word2Vec puede alcanzar cuando se entrena con miles de millones de tokens, sino el espacio concreto que surge de esta pequeña muestra.

Ese matiz es importante porque Word2Vec aprende del corpus que recibe. Si en el catálogo la palabra `silla` aparece sobre todo cerca de `gaming`, `reposabrazos` y `reclinable`, su vector reflejará principalmente ese contexto. No aprenderá una definición universal de *silla*, sino las asociaciones que resultan frecuentes dentro de esta colección.

Por tanto, los vecinos obtenidos sirven para observar dos propiedades al mismo tiempo: cómo palabras con contextos parecidos terminan próximas y hasta qué punto la geometría aprendida depende de la distribución de los datos. Un corpus más amplio, diverso y equilibrado produciría relaciones distintas y normalmente más estables.

## 4.2. GloVe: convertir coapariciones globales en geometría

GloVe parte de una visión algo distinta. En lugar de construir ejemplos locales y resolver una tarea predictiva, comienza agregando las coapariciones observadas en todo el corpus. La matriz $X_{ij}$ registra cuántas veces aparece la palabra $j$ dentro del contexto de la palabra $i$.

A partir de esa matriz, el modelo aprende vectores minimizando una regresión ponderada:

$$
J=
\sum_{i,j}
f(X_{ij})
\left(
\mathbf{w}_i^\top\widetilde{\mathbf{w}}_j
+b_i+\widetilde{b}_j
-\log X_{ij}
\right)^2.
$$

El objetivo es que el producto escalar entre dos vectores, junto con sus términos de sesgo, aproxime el logaritmo de su frecuencia de coaparición. El logaritmo comprime diferencias extremas entre conteos: pasar de una a diez apariciones importa más que pasar de diez mil a diez mil diez. La función $f(X_{ij})$ limita además la influencia de pares demasiado raros, que pueden ser ruidosos, y de pares extremadamente frecuentes, que podrían dominar el entrenamiento.

La intuición central de GloVe no se encuentra solo en los conteos absolutos, sino en sus **proporciones**. Una palabra asociada con frecuencia a `oficina`, `lumbar` y `ajustable` presenta un patrón distinto de otra que aparece junto a `comedor`, `madera` y `tapizada`, aunque ambas puedan referirse a tipos de silla. Esas diferencias entre distribuciones de contexto son las que el modelo intenta convertir en relaciones geométricas.

Word2Vec y GloVe llegan a sus vectores por caminos distintos. Word2Vec optimiza una tarea predictiva a partir de ventanas y muestras locales; GloVe trabaja con estadísticas de coaparición agregadas sobre todo el corpus. Uno aprende recorriendo ejemplos, mientras que el otro aproxima una estructura global ya resumida.

Pese a esa diferencia, ambos comparten dos limitaciones importantes para nuestro buscador. Cada palabra recibe un único vector estático, independientemente del contexto en el que aparezca, y ninguno produce directamente una representación para una consulta o una ficha completa. Para comparar secuencias hay que combinar después los vectores de sus términos mediante alguna heurística, como una media simple o ponderada, perdiendo parte del orden y de las interacciones entre palabras.

No entrenaremos GloVe sobre esta muestra mínima, porque una matriz de coapariciones tan pequeña ofrecería poco valor adicional. En su lugar, situaremos Word2Vec y GloVe en una tabla comparativa. La tabla resumirá qué señal utiliza cada modelo, cómo construye su objetivo y qué limitaciones conservan pese a sus diferencias.

In [35]:
classic_models = pd.DataFrame(
    [
        ["Word2Vec", "predicción de contexto", "palabra", "no"],
        ["GloVe", "coaparición global", "palabra", "no"],
        ["FastText", "contexto + subpalabras", "palabra", "parcial"],
    ],
    columns=["modelo", "señal", "unidad", "fuera_de_vocabulario"],
)
classic_models


,modelo,señal,unidad,fuera_de_vocabulario
0,Word2Vec,predicción de contexto,palabra,no
1,GloVe,coaparición global,palabra,no
2,FastText,contexto + subpalabras,palabra,parcial


## 4.3. FastText: representar una palabra mediante subpalabras

FastText conserva el objetivo predictivo de Word2Vec, pero cambia la forma de construir la representación de cada término. En lugar de asociar una palabra únicamente con un vector indivisible, la descompone en fragmentos de caracteres y combina la información de todos ellos.

Con marcadores de inicio y final, `silla` puede generar piezas como `<si`, `sil`, `ill`, `lla` y `la>`. La representación final se obtiene sumando los vectores de esos n-gramas y, durante el entrenamiento, también puede incluir un vector específico para la palabra completa.

Esta composición permite que términos relacionados por su forma compartan parte de la representación. `silla` y `sillas` reutilizan muchos fragmentos, y una variante como `ergonomikas` puede recibir un vector aunque nunca haya aparecido exactamente en el corpus. FastText no necesita encontrar la palabra completa en el vocabulario: puede sintetizarla a partir de las subpalabras que sí ha aprendido.

Esta propiedad resulta especialmente útil en catálogos, donde abundan las variantes morfológicas, los errores tipográficos, las referencias técnicas y las palabras compuestas. Un sistema basado únicamente en vectores completos perdería cualquier término fuera de vocabulario; FastText, en cambio, conserva al menos una señal derivada de su estructura ortográfica.

Sin embargo, poder generar un vector no equivale a comprender la palabra. FastText extrapola a partir de su forma. Si dos términos se parecen mucho en caracteres, pero expresan conceptos distintos, las subpalabras pueden acercarlos de manera engañosa. Del mismo modo, una palabra desconocida formada por fragmentos frecuentes puede recibir una representación plausible sin que el modelo haya observado nunca su significado real.

Por eso el ejemplo debe interpretarse como una prueba de robustez frente al vocabulario, no como evidencia de comprensión contextual. FastText mejora el tratamiento de palabras nuevas, pero sigue produciendo una representación estática: una misma forma recibe el mismo vector con independencia de la frase en la que aparezca.

Entrenaremos FastText con el mismo corpus utilizado para Word2Vec y solicitaremos un vector para `ergonomikas`, una forma ausente del vocabulario original. Como ambos modelos verán exactamente los mismos datos, la diferencia podrá atribuirse a la composición mediante subpalabras y no a cambios en el corpus.

In [36]:
fasttext_model = FastText(
    sentences=tokenized_products,
    vector_size=64,
    window=5,
    min_count=2,
    sg=1,
    negative=8,
    workers=1,
    seed=42,
    epochs=40,
)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [37]:
misspelled_word = "ergonomikas"
print("Word2Vec conoce la cadena:", misspelled_word in word2vec_model.wv)
print("FastText puede representarla:", misspelled_word in fasttext_model.wv)
fasttext_model.wv.most_similar(misspelled_word, topn=5)


Word2Vec conoce la cadena: False
FastText puede representarla: True


[('ergonomico', 0.9602410197257996),
 ('ergonomica', 0.9411661624908447),
 ('ergonomia', 0.9386089444160461),
 ('frambuesa', 0.7907050251960754),
 ('reposapies', 0.7812103629112244)]

# 5. Representaciones contextuales

Word2Vec, GloVe y FastText introducen relaciones que no aparecen de forma explícita en los conteos, pero comparten una limitación importante: cada palabra mantiene el mismo vector con independencia de la frase en la que aparezca. El siguiente paso consiste en abandonar esa representación fija y permitir que el contexto transforme el significado de cada aparición.

## 5.1. BERT: el vector de un token depende de toda la secuencia

BERT reemplaza los vectores estáticos por un Transformer bidireccional. Cada token comienza representándose como la suma de tres componentes: su embedding léxico, su posición dentro de la secuencia y, cuando corresponde, el segmento al que pertenece. Esa representación inicial atraviesa después varias capas de **self-attention**, donde cada posición puede incorporar información procedente del resto del texto.

Para cada token se calculan tres proyecciones: consulta, clave y valor. La atención se expresa como

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)V.
$$

La matriz $QK^\top$ mide hasta qué punto cada token debe atender a los demás. Tras aplicar `softmax`, esos pesos se utilizan para combinar sus vectores de valor. El resultado es que la representación final de una palabra ya no depende solo de su identidad, sino también de las palabras que la rodean.

En `banco de madera para el jardín`, el vector de `banco` incorpora señales de `madera` y `jardín`, por lo que se orienta hacia el significado de asiento. En `banco que concede hipotecas`, el mismo token queda condicionado por un contexto financiero. BERT no mantiene dos entradas distintas en un diccionario: construye una representación diferente para cada aparición.

El uso de múltiples cabezas de atención permite además observar la secuencia desde relaciones distintas. Una cabeza puede atender a dependencias sintácticas, otra a entidades relacionadas y otra a palabras que ayudan a desambiguar el significado. Las capas sucesivas combinan esas señales y producen representaciones cada vez más contextuales.

El BERT original se preentrena principalmente mediante **Masked Language Modeling**. Algunos tokens se ocultan y el modelo debe reconstruirlos utilizando la información disponible a ambos lados. Esa tarea obliga al encoder a aprender relaciones semánticas y sintácticas muy ricas, por lo que sus representaciones se adaptan bien a clasificación, extracción de entidades o preguntas y respuestas.

Sin embargo, ese entrenamiento no exige que dos frases relacionadas queden próximas bajo similitud coseno. Utilizar el token `[CLS]` o promediar los tokens de un BERT genérico permite obtener un único vector por secuencia, pero no garantiza que el espacio resultante sea adecuado para recuperación. El modelo ha aprendido a representar texto, no necesariamente a construir una geometría en la que una consulta y su producto relevante aparezcan cerca.

## 5.2. Sentence-BERT: entrenar explícitamente el espacio de frases

Sentence-BERT parte de una idea distinta a la de utilizar un BERT genérico y tomar sin más el vector de `[CLS]`. Aquí el objetivo de entrenamiento sí consiste en construir un espacio donde frases relacionadas queden cerca y frases distintas queden separadas.

La arquitectura utiliza dos ramas con pesos compartidos. Una codifica la consulta y otra el documento, pero ambas emplean exactamente el mismo encoder. Cada secuencia produce un vector contextualizado por token y, sobre esas representaciones, se aplica una estrategia de *pooling* —habitualmente la media— para obtener un único vector por consulta y otro por documento.

El entrenamiento se realiza con pares o tripletas. Los ejemplos positivos enseñan qué textos deberían aproximarse, mientras que los negativos obligan al modelo a distinguir candidatos incorrectos. En una pérdida contrastiva con negativos dentro del propio batch, para una consulta $q_i$ y su documento positivo $d_i$ puede minimizarse

$$
\mathcal{L}_i
=
-\log
\frac{\exp(s(q_i,d_i)/\tau)}
{\sum_j \exp(s(q_i,d_j)/\tau)}.
$$

El numerador recompensa que la consulta tenga una similitud alta con su documento correcto. El denominador hace que ese positivo compita contra los demás documentos del batch, que actúan como negativos. La temperatura $\tau$ controla cuánto se amplifican las diferencias entre similitudes: valores pequeños vuelven la competición más exigente y concentran más peso en los ejemplos difíciles.

La calidad de los negativos resulta especialmente importante. Un producto aleatorio y claramente ajeno a la consulta enseña poco, porque el modelo ya puede separarlo con facilidad. Los **hard negatives**, en cambio, son candidatos plausibles pero incorrectos: comparten categoría, vocabulario o atributos con el resultado relevante, aunque fallen en una condición decisiva. Esos ejemplos obligan al encoder a aprender fronteras mucho más finas.

Sentence-BERT se conoce como **bi-encoder** porque consulta y documento se procesan por separado. Esta separación tiene una ventaja operativa fundamental: los embeddings del catálogo pueden calcularse una sola vez, almacenarse en un índice y reutilizarse para todas las consultas. En tiempo de búsqueda solo hay que codificar la nueva consulta y compararla con los vectores ya disponibles.

Un **cross-encoder** funciona de manera diferente. Concatena la consulta y cada documento y permite que todos sus tokens interactúen mediante atención en cada capa. Esa interacción detallada suele producir una estimación de relevancia más precisa, pero impide precalcular una representación independiente de cada producto. Para un catálogo de un millón de elementos habría que ejecutar el modelo un millón de veces por consulta.

Por eso ambos modelos suelen ocupar etapas distintas del sistema. El bi-encoder recupera rápidamente un conjunto reducido de candidatos entre millones de productos; el cross-encoder reordena después unas decenas o unos cientos, donde su mayor coste ya resulta asumible.

Para hacer visible esta diferencia, estimaremos cuántas evaluaciones necesita un cross-encoder a medida que crece el catálogo y cuántos encodings requiere un bi-encoder cuando los documentos ya están precalculados. Así veremos que la separación entre ambas arquitecturas no es solo conceptual: determina directamente la escala a la que puede operar cada una.

In [38]:
catalog_sizes = np.array([1_000, 10_000, 100_000, 1_000_000])
architecture_frame = pd.DataFrame(
    {
        "productos": np.tile(catalog_sizes, 2),
        "codificaciones_online": np.concatenate(
            [np.ones(4), catalog_sizes]
        ),
        "arquitectura": ["Bi-encoder"] * 4 + ["Cross-encoder"] * 4,
    }
)


In [39]:
architecture_figure = px.line(
    architecture_frame,
    x="productos",
    y="codificaciones_online",
    color="arquitectura",
    markers=True,
    log_x=True,
    log_y=True,
)
architecture_figure.update_layout(
    title="Trabajo online conceptual por consulta",
    yaxis_title="Pares que deben codificarse online",
)
architecture_figure.show()


# 6. Modelos modernos de embeddings

Sentence-BERT establece la arquitectura que permite comparar secuencias completas de forma eficiente, pero no identifica un único modelo. A partir de esa idea han surgido familias entrenadas con más datos, más idiomas y objetivos cada vez más orientados a recuperación. Comenzaremos con un encoder que puede ejecutarse localmente y continuaremos después con servicios gestionados evaluados bajo el mismo protocolo.

## 6.1. Un encoder open-weight: multilingual E5

La familia E5 reformula numerosas tareas de NLP como pares de textos y utiliza aprendizaje contrastivo para organizar su espacio vectorial. En recuperación, esos dos textos no siempre desempeñan el mismo papel: la consulta suele ser breve y expresar una necesidad, mientras que el documento contiene una descripción más extensa del candidato.

Para hacer explícita esa asimetría, E5 antepone `query:` a las consultas y `passage:` a los documentos. Estos prefijos no son simples etiquetas pensadas para facilitar la lectura. Forman parte de la distribución con la que se entrenó el modelo y le indican qué función cumple cada secuencia. Omitirlos o intercambiarlos puede degradar la calidad del espacio resultante.

Utilizaremos `intfloat/multilingual-e5-small`, que produce embeddings de 384 dimensiones. Los vectores se han calculado previamente y se han normalizado con L2 para que la ejecución del notebook sea reproducible y la primera comparación no dependa de descargar el modelo. El script de generación se mantiene disponible y recoge exactamente el preprocesamiento y la plantilla utilizados.

Trabajar con embeddings precalculados evita repetir inferencia, pero introduce un riesgo menos visible: que los vectores estén correctamente generados y, aun así, queden asociados a los productos equivocados. Por eso debemos validar dos contratos antes de realizar ninguna búsqueda.

El primero es la alineación de IDs. El orden de los `product_id` almacenados junto a la matriz debe coincidir exactamente con el orden del DataFrame. Si ambas estructuras contienen los mismos elementos, pero en posiciones distintas, el cálculo vectorial seguirá funcionando y devolverá scores aparentemente razonables; el problema es que cada vector se traducirá al producto incorrecto.

El segundo contrato afecta a la representación. La dimensión de la matriz, la normalización y la distribución de normas deben coincidir con los metadatos declarados. Una forma inesperada puede indicar que se ha cargado otro modelo, mientras que normas alejadas de uno revelarían que los vectores no se normalizaron como estaba previsto o que el archivo ha sufrido alguna transformación.

Cargaremos ahora los tres conjuntos de embeddings E5: productos, consultas originales y paráfrasis. Junto a ellos leeremos sus metadatos y, antes de recuperar ningún resultado, comprobaremos IDs, dimensiones y normas. Estas validaciones evitarán que un índice desalineado o incompatible produzca rankings plausibles en apariencia, pero falsos en su correspondencia con el catálogo.

In [40]:
product_archive = np.load(project_root / "data/esci/e5_products.npz")
query_archive = np.load(project_root / "data/esci/e5_queries.npz")
semantic_archive = np.load(
    project_root / "data/esci/e5_semantic_queries.npz"
)

e5_product_ids = product_archive["product_ids"].astype(str)
e5_product_embeddings = product_archive["embeddings"]
e5_query_ids = query_archive["query_ids"]
e5_query_embeddings = query_archive["embeddings"]
e5_semantic_ids = semantic_archive["query_ids"]
e5_semantic_embeddings = semantic_archive["embeddings"]


In [41]:
expected_product_ids = products["product_id"].astype(str).to_numpy()
np.testing.assert_array_equal(e5_product_ids, expected_product_ids)

product_norms = np.linalg.norm(e5_product_embeddings, axis=1)
print("Shape:", e5_product_embeddings.shape)
print("Norma mínima:", product_norms.min())
print("Norma máxima:", product_norms.max())


Shape: (336, 384)
Norma mínima: 0.99999994
Norma máxima: 1.0000001


### La prueba decisiva: misma intención, vocabulario diferente

La comparación más reveladora aparece cuando mantenemos fija la intención y cambiamos la forma de expresarla. Recuperaremos productos para la pareja de consultas sobre televisores utilizando tanto TF-IDF como E5. La consulta literal parte con ventaja para el sistema léxico porque comparte con las fichas términos muy informativos como `television`, `28` y `pulgadas`.

La paráfrasis elimina buena parte de ese solapamiento. Expresa la medida en centímetros, introduce el contexto de uso y evita reproducir la formulación exacta del catálogo. La necesidad sigue siendo la misma, pero las pistas superficiales cambian. Esa diferencia permite observar hasta qué punto cada representación depende de compartir palabras.

E5 no consulta un tesauro ni ejecuta una conversión de unidades durante la búsqueda. El encoder procesa la secuencia completa y produce un vector en el que quedan distribuidas las relaciones aprendidas durante el entrenamiento. Si expresiones como *televisor pequeño*, *pantalla de unas setenta centímetros* y *televisión de 28 pulgadas* aparecen en contextos relacionados, el modelo puede situarlas en regiones próximas del espacio.

No existe, sin embargo, una coordenada aislada que signifique `28 pulgadas equivalen a 71 centímetros`. La relación emerge de la combinación de muchas dimensiones y no puede inspeccionarse como una regla explícita. Por eso el modelo puede aproximar bien esa equivalencia en algunos casos y fallar en otros: ha aprendido asociaciones distribucionales, no una calculadora fiable.

El score denso tampoco debe interpretarse como una probabilidad. Un coseno de 0,86 no significa que el producto tenga un 86 % de relevancia. Su función es ordenar candidatos dentro del mismo espacio y bajo la misma configuración. Comparar directamente ese 0,86 con el 0,72 producido por otro modelo carece de sentido si ambos generan distribuciones y geometrías diferentes.

Recuperaremos la misma pareja de consultas con TF-IDF y con E5 y mostraremos ambos rankings uno junto a otro. Los títulos, más que los scores aislados, permitirán juzgar qué sistema conserva mejor la intención cuando desaparece el vocabulario literal.

In [42]:
semantic_position = np.flatnonzero(
    e5_semantic_ids == demonstration_query_id
).item()
e5_semantic_query = e5_semantic_embeddings[semantic_position]
e5_semantic_scores = e5_product_embeddings @ e5_semantic_query


In [43]:
pd.concat(
    {
        "TF-IDF · paráfrasis": top_products(
            semantic_scores, query_id=demonstration_query_id
        ),
        "E5 · paráfrasis": top_products(
            e5_semantic_scores, query_id=demonstration_query_id
        ),
    },
    names=["sistema", "fila"],
)


product_id  \
sistema             fila               
TF-IDF · paráfrasis 0     B087QYF1WC   
                    1     B00KKFCLF2   
                    2     B07SWYRYB4   
                    3     B071DXH7YQ   
                    4     B087Q5SLQT   
                    5     B091T4NPSM   
E5 · paráfrasis     0     B07SWYRYB4   
                    1     B07CRRY36Y   
                    2     B07CQ84VKG   
                    3     B07CKWJK2B   
                    4     B07V1XCNVT   
                    5     B07YCNBYL2   

                                                              product_title  \
sistema             fila                                                      
TF-IDF · paráfrasis 0     Pipishell Baldas Pared, Estanteria de Pared de...   
                    1     SONGMICS Taburete de Bar Juego de 2, Sillas de...   
                    2     Vencipo Estantería Blanca para Fotos de Colgan...   
                    3     Zollner Taburete plegable multiusos, 30 cm altura   
                    4     Piorlado Baldas Pared Negro de 3 Niveles, Esta...   
                    5     DUFU Estanteria Ducha de Esquina para Baño Sin...   
E5 · paráfrasis     0     Vencipo Estantería Blanca para Fotos de Colgan...   
                    1     LG Consumer Electronics 28TK410V-WZ - Monitor ...   
                    2     LG Electronics 28TK410V-PZ - Monitor/TV de 28"...   
                    3     LG Electronics 24TK410V-PZ - Monitor/TV de 24"...   
                    4     LG 28TL510S-PZ - Monitor Smart TV de 70cm (28"...   
                    5     Televisor Led 24 Pulgadas HD, TD Systems K24DL...   

                         product_brand     score esci_label  
sistema             fila                                     
TF-IDF · paráfrasis 0        Pipishell  0.123402        NaN  
                    1         SONGMICS  0.105485        NaN  
                    2          Vencipo  0.103417        NaN  
                    3          ZOLLNER  0.102513        NaN  
                    4         piorlado  0.101748        NaN  
                    5             DUFU  0.098948        NaN  
E5 · paráfrasis     0          Vencipo  0.858186        NaN  
                    1               LG  0.855886          E  
                    2               LG  0.855205          E  
                    3               LG  0.854246          S  
                    4               LG  0.854096          E  
                    5       TD Systems  0.852109          S

## 6.2. Modelos de embeddings mediante API

Los proveedores gestionados eliminan la operación del modelo y ofrecen endpoints versionados, escalado y facturación por uso. A cambio, introducen dependencia externa, coste variable, límites de tasa, requisitos de privacidad y migraciones cuando un modelo se retira. La calidad no puede evaluarse leyendo una tabla comercial: todos los modelos se ejecutarán sobre los mismos textos, candidatos y métricas.

Las celdas siguientes realizan llamadas reales. Leen `OPENAI_API_KEY`, `COHERE_API_KEY` y `GEMINI_API_KEY` desde `.env`; no contienen interruptores adicionales. Se calculan embeddings de los 336 productos, las 12 consultas originales y las 8 paráfrasis. Cada proveedor recibe el rol correcto de consulta o documento cuando su contrato lo permite.

El coste total es pequeño para esta muestra, pero no es cero. En un catálogo real los documentos se embeben por lotes y se persisten; no deben recalcularse en cada ejecución analítica. También deben registrarse latencia, modelo, dimensión y fecha, porque una comparación sin configuración no es reproducible.

Antes de llamar a ningún proveedor, cargaremos el catálogo versionado de modelos y crearemos una estructura común para registrar embeddings, dimensión, tiempo y estado. Así todas las APIs desembocarán en el mismo protocolo de evaluación.


In [44]:
from vector_search_session.model_catalog import load_model_catalog

model_catalog = load_model_catalog()
provider_model_ids = {
    "openai-text-embedding-3-small",
    "openai-text-embedding-3-large",
    "cohere-embed-v4",
    "google-gemini-embedding-001",
    "google-gemini-embedding-2",
}
provider_rows = [
    model.as_table_record()
    for model in model_catalog.models
    if model.identifier in provider_model_ids
]
pd.DataFrame(provider_rows)[
    ["modelo", "estado", "modalidades", "contexto_tokens", "dimensiones"]
]


,modelo,estado,modalidades,contexto_tokens,dimensiones
0,text-embedding-3-small,available,text,8192,1-1536
1,text-embedding-3-large,available,text,8192,1-3072
2,embed-v4.0,available,"text, image, mixed-text-image",128000,"256, 512, 1024, 1536"
3,gemini-embedding-2,stable,"text, image, video, audio, pdf",8192,"768, 1536, 3072"
4,gemini-embedding-001,scheduled-retirement-2026-07-14,text,2048,"768, 1536, 3072"


In [45]:
from time import perf_counter

provider_document_texts = products["searchable_text"].tolist()
provider_original_texts = unique_queries["query"].tolist()
provider_semantic_texts = semantic_queries["semantic_query"].tolist()
api_timings = []


### OpenAI: `text-embedding-3-small` y `text-embedding-3-large`

Ambos modelos reciben texto mediante `client.embeddings.create`. `text-embedding-3-small` entrega 1.536 dimensiones por defecto y prioriza coste y velocidad. `text-embedding-3-large` entrega 3.072 y dispone de mayor capacidad. El parámetro `dimensions` permite solicitar una representación más corta entrenada con propiedades Matryoshka.

OpenAI devuelve vectores normalizados, por lo que coseno y producto escalar producen el mismo ranking. Aun así, la función normaliza de nuevo de forma defensiva después de validar la forma. No se mezclan vectores de `small` y `large`, aunque se solicite la misma dimensión: compartir shape no significa compartir espacio.

Para hacer comparable el coste de almacenamiento se solicitan 512 dimensiones en ambos modelos. Esta decisión no presupone que sea el mejor punto; forma parte del experimento y queda registrada en el nombre del sistema.

Crearemos el cliente de OpenAI y una función de batching reutilizable. Después calcularemos por separado documentos, consultas originales y paráfrasis para `small` y `large`, conservando la configuración exacta de cada ejecución.


In [46]:
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def embed_openai_batch(texts: list[str], model: str) -> np.ndarray:
    response = openai_client.embeddings.create(
        model=model,
        input=texts,
        dimensions=512,
        encoding_format="float",
    )
    vectors = np.asarray(
        [item.embedding for item in response.data], dtype=np.float32
    )
    return safe_l2_normalize(vectors, axis=1)


In [47]:
def encode_in_batches(
    texts: list[str], encoder, batch_size: int
) -> np.ndarray:
    batches = []
    for batch_start in range(0, len(texts), batch_size):
        text_batch = texts[batch_start : batch_start + batch_size]
        batches.append(encoder(text_batch))
    return np.vstack(batches)


In [48]:
started_at = perf_counter()
openai_small_products = encode_in_batches(
    provider_document_texts,
    lambda batch: embed_openai_batch(batch, "text-embedding-3-small"),
    128,
)
openai_small_original = embed_openai_batch(
    provider_original_texts, "text-embedding-3-small"
)
openai_small_semantic = embed_openai_batch(
    provider_semantic_texts, "text-embedding-3-small"
)
api_timings.append(
    ("OpenAI small · 512d", perf_counter() - started_at)
)


In [49]:
started_at = perf_counter()
openai_large_products = encode_in_batches(
    provider_document_texts,
    lambda batch: embed_openai_batch(batch, "text-embedding-3-large"),
    128,
)
openai_large_original = embed_openai_batch(
    provider_original_texts, "text-embedding-3-large"
)
openai_large_semantic = embed_openai_batch(
    provider_semantic_texts, "text-embedding-3-large"
)
api_timings.append(
    ("OpenAI large · 512d", perf_counter() - started_at)
)


### Cohere `embed-v4.0`: rol explícito y entrada multimodal

Cohere obliga a declarar `input_type`. Para recuperación, los documentos utilizan `search_document` y las consultas `search_query`. Esta distinción permite entrenar un espacio asimétrico: una ficha extensa y una consulta breve no tienen la misma distribución, aunque deban ser comparables.

`embed-v4.0` admite salidas de 256, 512, 1.024 o 1.536 dimensiones y entradas de texto, imagen o combinaciones. Aquí se eligen 1.024 dimensiones y embeddings `float`. La API limita cada llamada a 96 entradas, por lo que el catálogo se divide en lotes.

El tipo de entrada debe considerarse parte de la versión del índice. Embeber documentos como `search_query` no genera un error de shape: genera vectores válidos en una configuración equivocada, un fallo mucho más difícil de detectar.

La implementación enviará el catálogo como `search_document` y las dos colecciones de consultas como `search_query`. También dividirá las entradas en lotes para respetar el límite del endpoint y registrará el tiempo total de indexación.


In [50]:
from random import uniform
from time import sleep

import cohere

cohere_client = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

def embed_cohere_batch(texts: list[str], input_type: str) -> np.ndarray:
    response = cohere_client.embed(
        texts=texts,
        model="embed-v4.0",
        input_type=input_type,
        embedding_types=["float"],
        output_dimension=1_024,
    )
    vectors = np.asarray(response.embeddings.float_, dtype=np.float32)
    return safe_l2_normalize(vectors, axis=1)


def embed_cohere_batch_with_retry(
    texts: list[str],
    input_type: str,
    max_attempts: int = 6,
) -> np.ndarray:
    for attempt in range(max_attempts):
        try:
            return embed_cohere_batch(texts, input_type)
        except cohere.TooManyRequestsError:
            if attempt == max_attempts - 1:
                raise

            delay_seconds = min(70, 8 * 2**attempt) + uniform(0, 1)
            print(
                "Cohere ha alcanzado su límite de tokens. "
                f"Reintentando en {delay_seconds:.1f} segundos..."
            )
            sleep(delay_seconds)

    raise RuntimeError("No se pudo obtener el embedding de Cohere.")


def encode_cohere_documents(
    texts: list[str],
    batch_size: int = 32,
    pause_seconds: float = 8,
) -> np.ndarray:
    batches = []

    for batch_start in range(0, len(texts), batch_size):
        text_batch = texts[batch_start : batch_start + batch_size]

        batches.append(
            embed_cohere_batch_with_retry(
                text_batch,
                "search_document",
            )
        )

        if batch_start + batch_size < len(texts):
            sleep(pause_seconds)

    return np.vstack(batches)

In [51]:
started_at = perf_counter()
cohere_products = encode_cohere_documents(provider_document_texts)
cohere_original = embed_cohere_batch_with_retry(
    provider_original_texts, "search_query"
)
cohere_semantic = embed_cohere_batch_with_retry(
    provider_semantic_texts, "search_query"
)
api_timings.append(("Cohere Embed 4 · 1024d", perf_counter() - started_at))


### Google: `gemini-embedding-001` frente a `gemini-embedding-2`

Los dos modelos no son versiones intercambiables del mismo espacio. `gemini-embedding-001` es textual, acepta `task_type` como `RETRIEVAL_DOCUMENT` o `RETRIEVAL_QUERY` y genera un vector independiente para cada string de una lista. `gemini-embedding-2` es multimodal, amplía el contexto y sustituye `task_type` por instrucciones escritas en la entrada.

Con Embedding 2 hay un detalle especialmente importante: si se envían varias partes dentro de un único `Content`, el modelo puede agregarlas en un solo embedding multimodal. Para obtener un vector por producto se construye un objeto `Content` independiente para cada texto. Los documentos reciben una plantilla con título y texto; las consultas reciben una instrucción de recuperación.

Ambos modelos permiten controlar la dimensión de salida, pero no comparten espacio. `gemini-embedding-2` normaliza automáticamente las dimensiones reducidas; `gemini-embedding-001` exige normalizarlas manualmente cuando son inferiores a la salida completa. Migrar entre ambos obliga a recalcular documentos y consultas. Actualizar solo la consulta produciría vectores formalmente válidos que no pueden compararse con el índice anterior.

Construiremos dos rutas distintas porque los contratos de `001` y `2` no son intercambiables. Cada una generará documentos y consultas con su rol correspondiente, y cualquier indisponibilidad quedará registrada sin ocultar la llamada real.


In [52]:
from google import genai
from google.genai import types

google_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def google_values(response) -> np.ndarray:
    vectors = np.asarray(
        [item.values for item in response.embeddings], dtype=np.float32
    )
    return safe_l2_normalize(vectors, axis=1)


In [53]:
def embed_google_001_batch(
    texts: list[str], task_type: str
) -> np.ndarray:
    response = google_client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=768,
        ),
    )
    return google_values(response)


In [54]:
google_001_available = True
try:
    started_at = perf_counter()
    google_001_products = encode_in_batches(
        provider_document_texts,
        lambda batch: embed_google_001_batch(batch, "RETRIEVAL_DOCUMENT"),
        64,
    )
    google_001_original = embed_google_001_batch(
        provider_original_texts, "RETRIEVAL_QUERY"
    )
    google_001_semantic = embed_google_001_batch(
        provider_semantic_texts, "RETRIEVAL_QUERY"
    )
    api_timings.append(("Gemini 001 · 768d", perf_counter() - started_at))
except Exception as legacy_error:
    google_001_available = False
    print(f"gemini-embedding-001 no disponible: {legacy_error}")


In [55]:
def embed_google_2_batch(
    texts: list[str], input_type: str
) -> np.ndarray:
    prefix = (
        "task: search result | query: "
        if input_type == "query"
        else "title: product | text: "
    )
    contents = [
        types.Content(parts=[types.Part(text=prefix + text)])
        for text in texts
    ]
    response = google_client.models.embed_content(
        model="gemini-embedding-2",
        contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=768),
    )
    return google_values(response)


In [56]:
started_at = perf_counter()
google_2_products = encode_in_batches(
    provider_document_texts,
    lambda batch: embed_google_2_batch(batch, "document"),
    64,
)
google_2_original = embed_google_2_batch(
    provider_original_texts, "query"
)
google_2_semantic = embed_google_2_batch(
    provider_semantic_texts, "query"
)
api_timings.append(("Gemini 2 · 768d", perf_counter() - started_at))


## 6.3. Matryoshka y reducción de dimensionalidad

Algunos encoders modernos se entrenan mediante **Matryoshka Representation Learning**, una estrategia que busca que un mismo embedding siga siendo útil incluso cuando solo se conserva una parte inicial de sus dimensiones. Durante el entrenamiento, la pérdida se aplica no solo al vector completo, sino también a varios prefijos, como las primeras 256, 512 o 768 coordenadas.

La idea recuerda a una representación anidada: el vector completo contiene la versión más rica, pero sus primeros bloques ya deberían preservar una parte útil de la información. Esto permite elegir después un compromiso entre calidad y coste sin tener que entrenar un modelo distinto para cada dimensión.

La propiedad, sin embargo, no aparece automáticamente en cualquier embedding. Truncar un vector arbitrario puede destruir información importante si el modelo no fue entrenado para organizarla de esa forma. Por eso deben respetarse tanto el modelo como las dimensiones oficialmente soportadas por el proveedor.

Tras reducir la dimensión, algunas implementaciones requieren volver a normalizar los vectores con L2. También hay que mantener la configuración perfectamente alineada: consultas y documentos deben generarse con la misma dimensión, el mismo preprocesamiento y la misma política de normalización. Cambiar la longitud del vector implica además reconstruir el índice, porque su forma deja de ser compatible con los embeddings anteriores.

El ahorro de memoria puede calcularse de forma directa antes de considerar la sobrecarga del índice. Con `float32`, cada coordenada ocupa cuatro bytes, de modo que reducir la dimensión disminuye linealmente el tamaño bruto de cada vector. La cuantización puede llevar el coste aún más lejos mediante `float16`, `int8` o incluso representaciones binarias, aunque introduce un nuevo compromiso entre compresión y calidad.

Calcularemos ahora cuánto ocuparía almacenar un millón de vectores con varias dimensiones. El gráfico no determinará cuál es la configuración adecuada, pero permitirá visualizar el coste que después habrá que contrastar con métricas como recall y nDCG.

In [57]:
dimension_options = np.array([256, 384, 512, 768, 1_024, 1_536, 3_072])
storage_frame = pd.DataFrame(
    {
        "dimension": dimension_options,
        "GiB_por_millon": dimension_options * 4 * 1_000_000 / 2**30,
    }
)


In [58]:
storage_figure = px.bar(
    storage_frame,
    x="dimension",
    y="GiB_por_millon",
    text_auto=".1f",
)
storage_figure.update_layout(
    title="Almacenamiento bruto de un millón de vectores float32",
    yaxis_title="GiB sin contar el índice",
)
storage_figure.show()


## 6.4. Open-weight no significa una única familia

E5 es solo un punto del mapa. BGE-M3 puede producir representaciones dense, sparse aprendidas y late-interaction desde un mismo modelo. Qwen3-Embedding admite instrucciones y dimensiones Matryoshka. EmbeddingGemma prioriza despliegue compacto. La elección depende del idioma, longitud, licencia, hardware, modalidad y patrón de consulta.

Operar pesos propios evita enviar el corpus a una API y permite controlar versiones, cuantización y batching. A cambio, hay que aprovisionar memoria, servir el modelo, gestionar concurrencia, medir colas, desplegar actualizaciones y vigilar que tokenizer y checkpoint coincidan. `Open-weight` describe acceso al artefacto; no significa coste operativo cero ni licencia sin restricciones.

La comparación correcta no enfrenta `API` contra `open source` como etiquetas abstractas. Enfrenta configuraciones concretas: modelo, dimensión, plantilla, normalización, hardware, coste y calidad sobre las mismas consultas.


# 7. Más allá de un único vector denso

Los embeddings densos resultan atractivos porque condensan una consulta o un producto completo en un único vector. Esa representación permite comparar millones de elementos con gran eficiencia y captura relaciones semánticas que una coincidencia literal podría pasar por alto. Sin embargo, la misma compresión que hace posible una recuperación tan rápida también obliga al modelo a resumir toda la secuencia en un solo punto del espacio.

Durante ese proceso pueden diluirse detalles que dependen de interacciones muy concretas entre palabras. Una medida como *28 pulgadas*, una negación como *sin gluten* o la correspondencia entre varios atributos pueden quedar representadas de forma menos precisa que el significado general del texto. Dos productos pueden parecer globalmente similares y, aun así, diferir justo en la condición que determina si son válidos para la consulta.

Por eso la recuperación neuronal no se limita a elegir entre búsqueda léxica y un único embedding denso. Existen arquitecturas que conservan señales más interpretables, sistemas que combinan rankings independientes y modelos que mantienen representaciones separadas para cada token. Cada alternativa desplaza el equilibrio entre precisión, coste de almacenamiento, latencia y complejidad operativa.

## 7.1. Sparse aprendido y SPLADE

SPLADE ocupa una posición intermedia entre la recuperación léxica clásica y los embeddings densos. En lugar de representar cada texto mediante un vector corto y denso, utiliza un Transformer para producir un vector disperso definido sobre el vocabulario del modelo. La mayoría de sus dimensiones permanecen en cero, mientras que un conjunto reducido de términos recibe pesos positivos.

Esas dimensiones siguen correspondiendo a palabras interpretables. Podemos observar qué términos activa el modelo y con qué intensidad, como ocurriría en una representación basada en bolsa de palabras. La diferencia es que los pesos ya no proceden únicamente de la frecuencia literal. SPLADE aprende qué conceptos conviene activar para representar el significado del texto.

Esto permite realizar expansión semántica. Una consulta puede asignar peso a términos que no aparecen escritos, pero que el modelo considera relacionados con la intención. Del mismo modo, una ficha de producto puede activar vocabulario alternativo que facilite el encuentro entre expresiones distintas. El sistema conserva así parte de la transparencia y de la infraestructura de los índices invertidos, pero incorpora relaciones aprendidas que un método puramente léxico no conocería.

La ventaja operativa es que la representación sigue siendo sparse y puede aprovechar estructuras de recuperación basadas en postings. El coste es que generar esos vectores requiere ejecutar un modelo neuronal y que la expansión puede aumentar el número de términos activos por documento. La eficiencia final depende, por tanto, de cuánto consiga el entrenamiento mantener la dispersión sin perder capacidad semántica.

## 7.2. Recuperación híbrida

Otra estrategia consiste en no obligar a una única representación a resolver todos los tipos de consulta. Un sistema híbrido mantiene al menos dos recuperadores independientes: uno léxico, como TF-IDF o BM25, y otro denso basado en embeddings. Cada uno produce su propio ranking y aporta una señal distinta.

El recuperador léxico suele destacar cuando importan coincidencias exactas, números, marcas, modelos o términos poco frecuentes. El modelo denso, en cambio, puede recuperar productos relacionados por significado aunque apenas compartan palabras con la consulta. La fusión intenta conservar ambas fortalezas sin exigir que una sola puntuación represente todos esos comportamientos.

Un problema inmediato es que los scores no son directamente comparables. Un valor BM25 y una similitud coseno pertenecen a escalas distintas y pueden variar de manera desigual entre consultas. Sumarlos después de una normalización arbitraria puede hacer que uno de los sistemas domine por razones numéricas y no por calidad.

Reciprocal Rank Fusion evita ese problema trabajando con posiciones en lugar de scores. Para cada documento suma contribuciones de la forma

$$
\operatorname{RRF}(d)=\sum_r \frac{1}{k+r(d)},
$$

donde $r(d)$ representa la posición del documento en cada ranking y $k$ suaviza la ventaja de los primeros puestos. Un producto bien situado en ambos recuperadores acumula una puntuación alta, mientras que uno que aparece solo en una lista todavía puede aportar valor si ocupa una posición destacada.

La principal virtud de RRF es su robustez. No necesita calibrar escalas ni entrenar parámetros complejos y suele ofrecer una base sólida para sistemas híbridos. Una alternativa más flexible consiste en aprender la fusión mediante features como los scores originales, las posiciones, coincidencias de marca o señales de popularidad. Ese enfoque puede superar una regla fija, pero necesita datos de relevancia suficientes y una validación cuidadosa para no aprender sesgos propios del histórico.

## 7.3. Late interaction y ColBERT

ColBERT evita comprimir toda la consulta y todo el documento en un único vector. En su lugar, conserva una representación contextual para cada token. La consulta y el producto se codifican por separado, pero la puntuación final se calcula mediante interacciones entre sus tokens.

Para cada token de la consulta, ColBERT busca cuál de los tokens del documento ofrece la mayor similitud. Después suma esas mejores correspondencias:

$$
\operatorname{MaxSim}(Q,D)
=
\sum_i \max_j \mathbf{q}_i^\top \mathbf{d}_j.
$$

Esta operación permite que distintas partes de la consulta encuentren apoyo en lugares distintos del producto. El token asociado a una medida puede alinearse con la medida de la ficha, una restricción puede encontrar su expresión equivalente y otro término puede capturar la categoría general. La puntuación no depende de que toda esa información sobreviva dentro de un único resumen vectorial.

La interacción se denomina *late* porque la consulta y los documentos se codifican antes de encontrarse. Los vectores de los productos pueden precomputarse e indexarse, y solo durante la búsqueda se realizan las comparaciones token a token necesarias para construir MaxSim. Esto resulta mucho más eficiente que un cross-encoder que concatena cada consulta con cada documento y ejecuta el Transformer completo para cada pareja.

La mejora de granularidad tiene un coste evidente. En lugar de almacenar un vector por producto, hay que conservar varios vectores, normalmente uno por token. También aumenta el número de comparaciones necesarias durante la consulta y se requieren estructuras de índice específicas para recuperar eficientemente esas representaciones.

Antes de construir un sistema ColBERT completo, observaremos su mecanismo sobre un ejemplo pequeño. Generaremos una matriz de similitudes entre los tokens de una consulta y los de un producto, identificaremos el máximo de cada fila y sumaremos esas contribuciones. Así podremos ver no solo la puntuación final de MaxSim, sino también qué palabra del documento respalda a cada término de la consulta y qué partes quedan sin una correspondencia clara.

In [59]:
query_tokens = ["asiento", "trabajo", "espalda"]
product_tokens = ["silla", "oficina", "lumbar", "roja"]
token_similarities = np.array(
    [
        [0.88, 0.22, 0.34, 0.05],
        [0.18, 0.91, 0.25, 0.03],
        [0.30, 0.31, 0.86, 0.02],
    ]
)
maxsim_score = token_similarities.max(axis=1).sum()


In [60]:
maxsim_figure = px.imshow(
    token_similarities,
    x=product_tokens,
    y=query_tokens,
    text_auto=".2f",
    color_continuous_scale="Blues",
    zmin=0,
    zmax=1,
)
maxsim_figure.update_layout(
    title=f"Late interaction: MaxSim = {maxsim_score:.2f}"
)
maxsim_figure.show()


## 7.4. Embeddings multimodales

Hasta ahora toda la información ha sido textual. Un embedding **multimodal** amplía el espacio para representar imágenes, audio, vídeo u otros contenidos junto al texto. Esto permite que una consulta escrita recupere fotografías de producto o que una imagen encuentre fichas visualmente relacionadas.

CLIP es uno de los ejemplos más conocidos: entrena un encoder visual y otro textual para acercar cada imagen a su descripción correcta y separarla de las demás. Entre los servicios gestionados, Cohere `embed-v4.0` admite texto e imágenes, mientras que `gemini-embedding-2` incorpora texto, imagen, audio, vídeo y PDF dentro de un espacio compartido.

En un marketplace, esta capacidad aporta valor cuando el color, el patrón, la forma o el acabado aparecen en la fotografía pero no en la descripción. Añadir imágenes no garantiza una mejora: modifica el contrato del índice y obliga a evaluar consultas en las que la señal visual sea realmente necesaria.


# 8. Evaluación del buscador

Para cada consulta ESCI solo se conocen juicios sobre un conjunto de candidatos. La evaluación se realiza dentro de ese conjunto. Buscar sobre los 336 productos es útil para inspección, pero no permite declarar irrelevante cualquier producto sin etiqueta. Confundir `no juzgado` con `irrelevante` sesga las métricas.

## 8.1. Recall@k: comprobar la cobertura de la recuperación

La primera pregunta es si los productos relevantes aparecen dentro de los primeros $k$ candidatos. **Recall@k** mide qué proporción del conjunto relevante ha sobrevivido a esa primera selección:

$$
\operatorname{Recall@k}
=\frac{|\operatorname{Rel}\cap\operatorname{Top}_k|}
{|\operatorname{Rel}|}.
$$

En este ejercicio consideraremos relevantes las etiquetas `Exact` y `Substitute`. `Complement` no resuelve por sí solo la necesidad, aunque nDCG conservará su pequeña ganancia graduada. Esta decisión debe hacerse explícita: cambiar qué etiquetas forman $\operatorname{Rel}$ cambia la pregunta que responde la métrica.

Recall@k resulta decisivo en una arquitectura de dos etapas. Si el recuperador deja fuera un producto, ningún reranker podrá devolverlo después. La métrica no valora el orden dentro del top-$k$; solo comprueba que la primera etapa ha proporcionado cobertura suficiente.

## 8.2. nDCG: valorar el orden y la relevancia graduada

Una vez comprobada la cobertura, necesitamos medir cómo se ordenan los candidatos. ESCI asigna ganancias `E=1`, `S=0.1`, `C=0.01`, `I=0`. DCG descuenta logarítmicamente la ganancia según la posición:

$$DCG=\sum_{r=1}^{n}\frac{gain_r}{\log_2(r+1)}.$$

El ranking ideal coloca primero todos los Exact, después Substitute, Complement e Irrelevant. Dividir por su DCG produce nDCG entre 0 y 1. La métrica premia especialmente los primeros puestos, donde se concentra la atención del usuario, y permite comparar consultas con distinto número de candidatos.

Se calculará una puntuación por consulta y después una media macro: cada consulta pesa lo mismo. En producción podrían añadirse pesos por frecuencia o valor, pero entonces la métrica respondería a otra pregunta.

Implementaremos primero recall@10 y nDCG como funciones independientes. Después crearemos un registro común por modelo, consulta y tipo de entrada, de manera que la misma evaluación sirva para TF-IDF, E5 y los proveedores gestionados.


In [61]:
def esci_ndcg(labels: pd.Series, scores: np.ndarray) -> float:
    gains = labels.map(ESCI_GAINS).to_numpy(dtype=float)
    ranked_gains = gains[np.argsort(-scores, kind="stable")]
    discounts = np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum(ranked_gains / discounts)
    ideal_dcg = np.sum(np.sort(gains)[::-1] / discounts)
    return float(dcg / ideal_dcg) if ideal_dcg else 0.0


def esci_recall_at_k(
    labels: pd.Series,
    scores: np.ndarray,
    k: int = 10,
) -> float:
    relevant = labels.isin(["E", "S"]).to_numpy()
    relevant_total = int(relevant.sum())
    if relevant_total == 0:
        return 0.0
    top_k = np.argsort(-scores, kind="stable")[:k]
    return float(relevant[top_k].sum() / relevant_total)


In [62]:
product_position = {
    product_id: position
    for position, product_id in enumerate(products["product_id"])
}
original_query_position = {
    int(query_id): position
    for position, query_id in enumerate(unique_queries["query_id"])
}
semantic_query_position = {
    int(query_id): position
    for position, query_id in enumerate(semantic_queries["query_id"])
}


In [63]:
def judged_candidates(query_id: int) -> tuple[pd.DataFrame, np.ndarray]:
    query_group = judgments.loc[judgments["query_id"] == query_id]
    positions = np.array(
        [product_position[item] for item in query_group["product_id"]]
    )
    return query_group, positions


In [64]:
def evaluation_record(
    model_name: str,
    query_id: int,
    query_type: str,
    labels: pd.Series,
    scores: np.ndarray,
) -> dict[str, object]:
    return {
        "modelo": model_name,
        "query_id": query_id,
        "tipo": query_type,
        "Recall@10": esci_recall_at_k(labels, scores, k=10),
        "nDCG": esci_ndcg(labels, scores),
    }


In [65]:
def evaluate_dense_view(
    model_name: str,
    document_embeddings: np.ndarray,
    query_ids: np.ndarray,
    query_embeddings: np.ndarray,
    query_type: str,
) -> list[dict[str, object]]:
    rows = []
    for query_id, query_vector in zip(
        query_ids, query_embeddings, strict=True
    ):
        query_group, positions = judged_candidates(int(query_id))
        scores = document_embeddings[positions] @ query_vector
        rows.append(
            evaluation_record(
                model_name,
                int(query_id),
                query_type,
                query_group["esci_label"],
                scores,
            )
        )
    return rows


In [66]:
def evaluate_dense_system(
    model_name: str,
    document_embeddings: np.ndarray,
    original_embeddings: np.ndarray,
    semantic_embeddings: np.ndarray,
) -> list[dict[str, object]]:
    original_rows = evaluate_dense_view(
        model_name,
        document_embeddings,
        unique_queries["query_id"].to_numpy(),
        original_embeddings,
        "original",
    )
    return original_rows + evaluate_dense_view(
        model_name, document_embeddings,
        semantic_queries["query_id"].to_numpy(),
        semantic_embeddings, "paráfrasis",
    )


In [67]:
def evaluate_sparse_view(
    query_frame: pd.DataFrame,
    text_column: str,
    query_type: str,
) -> list[dict[str, object]]:
    rows = []
    for query_row in query_frame.itertuples(index=False):
        query_id = int(query_row.query_id)
        query_group, positions = judged_candidates(query_id)
        query_text = getattr(query_row, text_column)
        query_sparse = tfidf_vectorizer.transform([query_text])
        scores = (
            tfidf_matrix[positions] @ query_sparse.T
        ).toarray().ravel()
        rows.append(
            evaluation_record(
                "TF-IDF",
                query_id,
                query_type,
                query_group["esci_label"],
                scores,
            )
        )
    return rows


In [68]:
def evaluate_tfidf_system() -> list[dict[str, object]]:
    original_rows = evaluate_sparse_view(
        unique_queries, "query", "original"
    )
    semantic_rows = evaluate_sparse_view(
        semantic_queries, "semantic_query", "paráfrasis"
    )
    return original_rows + semantic_rows


In [69]:
evaluation_rows = evaluate_tfidf_system()
evaluation_rows.extend(
    evaluate_dense_system(
        "multilingual-e5-small",
        e5_product_embeddings,
        e5_query_embeddings,
        e5_semantic_embeddings,
    )
)
local_evaluation = pd.DataFrame(evaluation_rows)


In [70]:
local_summary = (
    local_evaluation.groupby(["modelo", "tipo"], as_index=False)[
        ["Recall@10", "nDCG"]
    ]
    .mean()
    .sort_values(["tipo", "nDCG"], ascending=[True, False])
)
local_summary


,modelo,tipo,Recall@10,nDCG
2,multilingual-e5-small,original,0.535295,0.833430
0,TF-IDF,original,0.493893,0.758592
3,multilingual-e5-small,paráfrasis,0.436582,0.822395
1,TF-IDF,paráfrasis,0.454680,0.778800


In [71]:
semantic_comparison = (
    local_evaluation.loc[local_evaluation["tipo"] == "paráfrasis"]
    .pivot(index="query_id", columns="modelo", values="nDCG")
    .reset_index()
    .merge(
        semantic_queries[["query_id", "semantic_query"]],
        on="query_id",
    )
)
semantic_comparison["delta_E5"] = (
    semantic_comparison["multilingual-e5-small"]
    - semantic_comparison["TF-IDF"]
)


In [72]:
semantic_figure = px.bar(
    semantic_comparison.sort_values("delta_E5"),
    x="delta_E5",
    y="semantic_query",
    orientation="h",
    color="delta_E5",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
)
semantic_figure.update_layout(
    title="Cambio de nDCG al sustituir TF-IDF por E5",
    xaxis_title="nDCG(E5) - nDCG(TF-IDF)",
    yaxis_title="Paráfrasis",
    height=620,
)
semantic_figure.show()


La comparación local separa dos fenómenos. En consultas originales, TF-IDF puede ser muy competitivo porque los usuarios del dataset suelen escribir nombres de categoría y atributos. En las paráfrasis, el solapamiento superficial se reduce de forma deliberada y E5 debería perder menos calidad.

El gráfico impide reducir la conclusión a `dense gana`. En esta muestra E5 mejora con claridad las necesidades expresadas como problema de espalda y cambio de unidad, pero pierde en varias paráfrasis de compatibilidad o restricción. Puede haber ambigüedad en la paráfrasis, información insuficiente en la ficha o una frontera que el encoder no aprendió bien. Cada barra negativa es un caso que debe inspeccionarse antes de desplegar.

Si el dense mejora las paráfrasis pero empeora medidas, negaciones o modelos, el siguiente experimento razonable es híbrido. Si una API mejora unas milésimas a cambio de multiplicar coste y latencia, puede no justificar la migración. Si un modelo grande no supera a uno pequeño en este slice, la dimensión extra no aporta valor demostrado.

Las celdas siguientes incorporan los cinco modelos API al mismo DataFrame. No se comparan sus cosenos crudos; se compara el ranking producido por cada espacio mediante nDCG.

Incorporaremos ahora los resultados disponibles de las APIs al mismo DataFrame. Las llamadas que no hayan podido ejecutarse no se convertirán en ceros: simplemente quedarán fuera de la comparación para no confundir indisponibilidad con mala calidad.


In [73]:
api_evaluation_rows = []
api_systems = [
    ("OpenAI small · 512d", openai_small_products, openai_small_original, openai_small_semantic),
    ("OpenAI large · 512d", openai_large_products, openai_large_original, openai_large_semantic),
    ("Cohere Embed 4 · 1024d", cohere_products, cohere_original, cohere_semantic),
    ("Gemini 2 · 768d", google_2_products, google_2_original, google_2_semantic),
]
if google_001_available:
    api_systems.append(
        ("Gemini 001 · 768d", google_001_products, google_001_original, google_001_semantic)
    )
for system_name, document_vectors, original_vectors, semantic_vectors in api_systems:
    api_evaluation_rows.extend(
        evaluate_dense_system(
            system_name,
            document_vectors,
            original_vectors,
            semantic_vectors,
        )
    )


In [74]:
full_evaluation = pd.concat(
    [local_evaluation, pd.DataFrame(api_evaluation_rows)],
    ignore_index=True,
)
full_summary = (
    full_evaluation.groupby(["modelo", "tipo"], as_index=False)["nDCG"]
    .mean()
    .sort_values(["tipo", "nDCG"], ascending=[True, False])
)
full_summary


,modelo,tipo,nDCG
4,Gemini 2 · 768d,original,0.882009
6,OpenAI large · 512d,original,0.852068
0,Cohere Embed 4 · 1024d,original,0.844113
2,Gemini 001 · 768d,original,0.842989
12,multilingual-e5-small,original,0.833430
8,OpenAI small · 512d,original,0.826642
10,TF-IDF,original,0.758592
5,Gemini 2 · 768d,paráfrasis,0.883434
3,Gemini 001 · 768d,paráfrasis,0.857665
7,OpenAI large · 512d,paráfrasis,0.852201


In [75]:
comparison_figure = px.bar(
    full_summary,
    x="modelo",
    y="nDCG",
    color="tipo",
    barmode="group",
    text_auto=".3f",
)
comparison_figure.update_layout(
    title="Calidad en consultas originales y paráfrasis",
    xaxis_tickangle=-25,
    height=580,
)
comparison_figure.show()


In [76]:
latency_frame = pd.DataFrame(
    api_timings, columns=["modelo", "segundos_embedding_muestra"]
)
latency_frame["dimension"] = latency_frame["modelo"].str.extract(
    r"(\d+)d"
).astype(int)
latency_frame


,modelo,segundos_embedding_muestra,dimension
0,OpenAI small · 512d,6.422894,512
1,OpenAI large · 512d,2.921166,512
2,Cohere Embed 4 · 1024d,92.712944,1024
3,Gemini 001 · 768d,9.371892,768
4,Gemini 2 · 768d,8.479491,768


## 8.3. Latencia, memoria y coste

Los tiempos anteriores son medidas de pared sobre una ejecución pequeña. Incluyen serialización, red, colas del proveedor y batching; no son un benchmark estable ni deben utilizarse como SLA. Una comparación operativa repetiría ejecuciones, separaría indexación de consulta, registraría percentiles y controlaría región, concurrencia y reintentos.

Tampoco se fijan precios en el notebook porque cambian con el tiempo y con el contrato. El coste debe calcularse con el uso medido y la tarifa vigente en el momento de la decisión. Lo reproducible aquí es el número de textos, la configuración del modelo y el tiempo observado, no una estimación monetaria congelada.


## 8.4. Análisis por consulta: la media no explica el fallo

Una media macro puede ocultar que un sistema gana ligeramente en siete consultas y falla de forma grave en una consulta crítica. Por eso se inspecciona el delta por `query_id` y el ranking concreto.

En el conjunto original, `cámaras bridge baratas` tiende a favorecer a TF-IDF: contiene una categoría precisa y el dense puede acercar accesorios o cámaras relacionadas. `sillas oficina ergonomicas` y su paráfrasis permiten observar el caso contrario. La representación semántica puede conectar el problema de espalda y la jornada larga con atributos ergonómicos que no aparecen literalmente en la consulta.

El análisis de errores debe convertir observaciones en nuevos slices: medidas y unidades, negaciones, compatibilidad, marcas, consultas por problema, complementos y resultados sin juicio. Un benchmark propio mejora cuando cada fallo recurrente se transforma en una prueba estable.

Cerraremos la evaluación construyendo una tabla por consulta con TF-IDF y E5. Ordenar por diferencia hará aparecer primero los fallos más graves y nos permitirá leer la media como el resumen de casos concretos, no como una conclusión aislada.


In [77]:
local_pivot = local_evaluation.pivot_table(
    index=["query_id", "tipo"],
    columns="modelo",
    values="nDCG",
).reset_index()
local_pivot["delta_E5"] = (
    local_pivot["multilingual-e5-small"] - local_pivot["TF-IDF"]
)
local_pivot.sort_values("delta_E5", ascending=False)


modelo,query_id,tipo,TF-IDF,multilingual-e5-small,delta_E5
19,101352,paráfrasis,0.634648,0.907658,0.273010
12,93437,original,0.738521,0.948170,0.209648
18,101352,original,0.812925,0.999281,0.186356
13,93437,paráfrasis,0.803029,0.978290,0.175262
1,18868,original,0.696457,0.831473,0.135017
16,100455,original,0.413354,0.502657,0.089303
0,13357,original,0.542945,0.623545,0.080600
14,96202,original,0.870704,0.933710,0.063006
7,38249,paráfrasis,0.871813,0.933628,0.061815
6,38249,original,0.930103,0.987312,0.057209


# 9. Selección de la arquitectura

MTEB y MMTEB son útiles para reducir una lista de cientos de modelos. Agregan tareas, idiomas y dominios bajo protocolos comunes. No sustituyen la evaluación del buscador: el catálogo, las consultas, la relevancia, la longitud, el idioma y la infraestructura pueden diferir de los benchmarks públicos.

Una decisión defendible combina al menos:

1. calidad media y por slices en juicios propios;
2. coste de indexación y coste por consulta;
3. latencia de codificación y recuperación;
4. memoria del vector y del índice;
5. privacidad, licencia y dependencia del proveedor;
6. estrategia de versionado y migración.

Para este marketplace, la hipótesis de salida más razonable no es eliminar TF-IDF. Es mantener una señal léxica para referencias y restricciones, añadir el mejor dense validado para consultas por necesidad y medir una fusión híbrida. Solo después tendría sentido experimentar con reranking, late interaction o multimodalidad.

El aprendizaje central puede resumirse con una cadena causal: **el texto de entrada determina la representación; el entrenamiento determina la geometría; la métrica convierte esa geometría en ranking; los juicios de negocio determinan si el ranking es bueno**.
